# GeoLifeCLEF v35 — full-set balanced attention

**Candidate, not proven improvement or SOTA.** V34 regressed to 0.24203 public /
0.21483 private. Keep the best v32 (0.24225 / 0.21514) until a new actual score wins.

## Run

1. Attach **GeoLifeCLEF25 @ CVPR & LifeCLEF** (`geolifeclef-2025`), official input only.
2. Select **T4 x1**, restart, **Run All**. Internet off is fine; no extra dataset.
3. Submit **only** `v35_export/GLC25_PA_submission_v35.csv` when
   `eligible_for_submission` is true. If `NO_SUBMISSION.json` appears, do not submit
   any diagnostic file; retain v32. Nothing is uploaded/submitted automatically.

## The change

V34 passed development by only +0.000430 F1; its bootstrap lower bound was
0.000000427. That weak reused signal did not transfer. V35 abandons graph reranking
and the permanent v29/v32 count/prefix lock. It predicts complete species sets.

Three attention models train for up to **48 epochs**. EMA checkpoints at 12/24/36/48
are selected using selection-only ranking F1 (with country/outside-core components).
The first model extends the successful geo-ASL recipe. Two new sensor-balanced
models separately pool Sentinel, Landsat, climate, environmental and static tokens,
then fuse them. They use geo/no-geo diversity, a training-defined rare residual
head, and a small per-survey normalized positive-ranking auxiliary objective.
All 5,016 species remain in the classifier/loss. No external pretrained weights.

Calibration chooses one of 15 probability-ensemble/count policies: equal,
ecology-lean or sensor-balanced pair, crossed with 3 expected-F1-surrogate count
scales or fixed top18/top24. Counts remain within 8..40 but are no longer inherited
from v29. Checkpoint choice and probability calibration use selection rows only.
Production uses all PA rows; selected epoch counts retain the development
learning-rate schedule prefix rather than silently compressing it.

## Spend the budget on new models

The historical matched-v32 development scores are embedded as a small verified
cache. Before trusting it, the notebook checks **official training metadata,
all four role ID hashes in both folds, all PA labels and species order**, and
recomputes cached calibration F1.
No old reference neural fits are repeated: **6 new development fits + at most
3 production fits**, not 22 old-reference fits before the new work.

This cache contains PA development results, not test labels or pretrained models.
The reference is the unchanged v32 recipe refit during v34, not the v34 graph result.
Calibration comparison uses the 1,500 saved views per fold (some survey IDs repeat
across folds and are deduplicated for country-support checks). Assessment covers
the complete 31,570 historical spatial-check surveys.

## Stricter promotion, honest failure output

Require calibration and assessment gain >=0.001, outside-DK/NL gain >=0.0005,
positive gains in both folds and country-macro, and assessment bootstrap lower
bound >=0.00025. These margins were introduced **after observing v34's failure**;
they are safeguards, not a new independent statistical guarantee.
If rejected, report the actual attempted model's scores rather than hiding them
behind an identical control with zero delta.

All 88,987 PA IDs have already been assessed. Countries poorly represented or
absent from PA remain a major limitation (especially Ukraine/UK). These checks
are **repeated development**, not a fresh holdout. Full-set replacement can regress
even when these gates pass. The pooling/loss/schedule/decoder changes are a joint
hypothesis, not a completed causal ablation or exact Bayes-optimal F1 algorithm.

## Runtime/output

Cooperative **10.75-hour guard**, whole remaining-plan admission with safety
margins, GPU preflight before loading predictors. Full v35 T4 duration is unmeasured;
there is no absolute 12-hour guarantee against stalled I/O/hardware.
Exactly five final files, <=16 MB: eligible CSV OR NO_SUBMISSION.json, report,
manifest, per-survey diagnostic CSV, and sampled individual-model top128 scores
with full probability mass and truth for both calibration and assessment.
Temporary feature caches/checkpoints are removed. Failures revoke eligibility.

The distinction between marginal-probability ranking and F1-optimal set decisions
is discussed by [Dembczynski et al., ICML 2013](https://proceedings.mlr.press/v28/dembczynski13.html).
Our count decoder is a ratio-of-expectations surrogate, not their exact algorithm.


In [ ]:
"""v35: full-set, checkpoint-selected attention ensemble; cached v32 comparator.

All development labels were consumed before. No independent performance claim.
"""
from __future__ import annotations

import base64
import copy
import hashlib
import json
import lzma
import math
import os
from pathlib import Path
import time
import traceback

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

import types as _types
_v27_source = lzma.decompress(base64.b64decode('...'))
assert hashlib.sha256(_v27_source).hexdigest() == '69a2d3952fa6324d9a05f4edb6fa7662388a198df89c407f7f034b94099f1dc0'
_v27 = _types.ModuleType('v35_frozen_v27')
exec(compile(_v27_source, 'frozen-v27', 'exec'), _v27.__dict__)
del _v27_source
_v29_source = lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4LmJNRhdABFoGYZDkepR8662PQUn4xZ6XcmQ/hL/kvr9CZNu7OZ7BTsxRAsOYMBV+kEfAlVaRLuDeekbHjOtmuhTxBdCOe6HZJxwqqDt9qVrPyu5X/6NGy0QdJbngcotqULQFfacScBNbbdOSStqetO64QR/vO+S7bUNqUujgjGz6OwXn8MwHMXx8+dfACclTWCPZVIHtlChHwzHKt6Y4ruUbOhXdtRYx9dJxcG0YkGGDH3GOx26MR+2b4jmjGZJgwPkbQZkg9SQnr4jhhslVFkUbM5afMmX7W0IB6Ewk/S5PZgJ1SC23uplBm9knz11eO7XQAi3M5vScHP9bMPRmdQmfc8LLFNU5urMKu8F7fxjOfqWz61KT7p/B7DyxqthdRmp2OS49OYZjVzNtiUHL4MBBrjuQWPOa/mduQ0BKN7Zv4ZRQwM/ufnBCTBREOaEMmbxG8PQxQH++E5A9GlVRengBZdVr5BibVTWhP8NIL2mNSGSft2Z8wKd1it0TdhTmiW7LduiasxqQyMt6ORk5QdGzdpkhnHPDQrQ2ZR9K1gzycBeAGInDTh5TmGesWv5U4WFhVXbDj96/SELUnFgscDQo9FhX32osEXOTrlFLTKorQ5RUisOb229ARqdhRDkYgVsFuxJ3cEtLOab7wwVhAKY9teIDIB+wGNKtSDNoRHSlfA5bbBJSHws1IpXfL7PfoaVX6gg1NZj80nzWcK+1md1+jdo30JOx9ocDo7t4evgHZjqMOiOhHfWzCRapVTJ+DaU+/IcoIxWQmqH3mvSwW9FcxwK0KSV8I1D2+94pR4feMHmmZYNfu5TEuSiDw5wAxoZzwsed0hyv3rx9JmL+brlPL1PwRW9vsy/ysc3ebWwish0hRNdl9cDs3Dx/G/bCVU8i/68DhYJNbTWJvVjA7j+waw4s5Zks/1OrIuiDIhx0T68q645m0Djti81Ezr5L6UqCq2M7iNAawqJpbU87D9ozuYun5J0wTGdf8FvjxbvzXBxQPXPxlsYYGiiTEakM2Xqhw55lk9f8r5CGh2DaisljGM39V2+/gmaFVDxFMuF2dCu2eVgEpBimQC9D7t5BsUqj6r30Sh6VZmNZzCq6r5wGXOpps7oYCicggl71E5uD85L767gtSZHn3s2TqKmoRyJaoHFfxvalcV3baX0ng3XPi5Snaivi0KK/K9cJVzUUhZGdFZ70Bj2uDsOOlIevgVk6GuGln6EFr0Uq8re9hh6SbVcW51ODb+2EvVWa0c1bNUntNNKX/Q1FqhKzny67Nzu2IDaiCz3tIgp8zp1ASB6/NHvkNHpxpLxl0pXkcWq+JeVyam3ZLYE3zZeFiT/o+wW67F5h8uGookMexDdyiVxvQnG7x13/Xl2zIRNqG6PWEy58VzfQgD7uQ1JVcjMocJwJZChJ4Fcc7L7LBf0O09Zrua3jHgdX7pfHq1URJempsEjb+abyJWglpIZvKD37oB1+l+rjgyY1tuMq+CKKw2BaAk6WnW6zcQBKAt5EvNMUBolL4Ou29La0McA8XT5H+8yVeObyLPUayQ2oMJVcREItshLXAmKRcM6pawihm7fj7Djqhcj1cUL/2IJdZuxlfF8t7KIcWeXCvWppm5Pe+W1T31EXoCqulGa2fO5t+ERM+uRefl8nUx/iTHtSJjENjfAryvjH668TftvvCPOVqWp8cqr0CJaETBe9ye3iT5YXbNHwx9WRgcfbmA71iyNbGTtIzINlnY4pol9FrZEfSkSQSL+RDZdkrVrCJO2SjOQxbSOlfmhAais47Bax2IiYmOpRx2/0IIeb51ZPcumd/P0+cjJesRtnPaUcLSiK3Lrrr9x0trgUoXfixns9vlLUrYVvPv+PQngBXQmdhMkxjan6Ic8boh82mpOAa2FYnXaxQ/aDvZVx/UiQIFR1okDsFpEMLZo2tLMpv30Hb/bbUDjAB3ufU5EkW2MxU4pyxasVxmpl0C8sGAHY4WHOhX2W8jxG33d3N7WqQoGy17tnPEtkyuvz9IXcoFxFxyg3zo017B8L57TCYKPN/UPJBr6974K86LPkwLU2mtNtm4pXxsDmguv+JtumwjK2WD02J1qfyI1kYfkN5kxYfbCB2ZjmSD0ZtyyB/1zdqPHyAQKjgVRLBt+R6MtQT0ek9LSZ6TWswg2wkR1JCxhJuSLhANQOWznTaf3oLbljc49U4z419PZkkegH433FRGJkpf6UwEmEzDIL3qoxAX8UOqzTvibEJOBpCFPkZ0IAacKQ1VYg9gz+Z+BTmh2NDRmcFtTDOLqSpJpJIXIpxWhPFQ5cMamVaBY34rucBB3tAMYWJKFl5kl5tmbDpYnOLxGHdY2WAybjidDJ4Vkq2o83MP3vmLzKz6MfoKgtsv3tw4BeiUcrXLUs93TCpUxjSiSXwP130mnu8d4gE0Zp8B87qd5BQfXRO+ml8s+QU+faCl8xvEWjkCYlZA6QKQd+OaHfrDD6cdi5H66CPmjHUH+kTw4CpXpYfcg3o3EFKp+hfHpGVb7TVVxbI16bV50Wu7kUIZv/W2S1JtUGhiF4xuqasLLXW+ngUaAt3IdZS6nJUd4qZXbJ/gdp1a9vZ9vhBZcAVgiTJSyt4hZw4Ze6syfZws1Ice5KeRvLKVT+m+LJv8d33W5NRPyb/E1SVOiV7bZqQVOQCm096Fm23/3OsX5wX3bR6XMCoL86ORqEnj6oPkgTC6z6JJjKuLTsKjV8u9L/8mhXxECLUCHtOnf++EZwkt0uVAQWagr8sXkiQWya9xz/3CgaE15i5r3EjUrJxdjrEobvcx1biGYPwEwUhpDdEL+uoe+2WYAIDLYODNDRy/X/PclMR/hk9OfmeZuh6gCZgAyqR0fc6CnnMjkF27iXOmEVL7nKtEBch5dVZnT+17dY45WuStXagSuvwyGx9/Qth9AUVN78rGwKykqUbKXvAqG6KxVxfIZNBqt/mvNxbSCgOJBCbsAw48ZrmumDnJXS0hNEL1EPpvCklDGp1mvd/S/arZFfWlmNasmNOJMIUWLZPjgtbgMa8n7LzDqvBKGY7+5LqFb+CCifezZQf/fBaO/isvk8XbFn4zg/g9JwYirWS+BtkrU23RgzvsG13Zcc/VwpIZT1Q/W6YuPROm4YxkBM7BmX6U7mFyUQ/FTXdptZF9RZYryb/6ro2rtl6hqqlTj40R70MowghC+1Ru/QF2VW787hzdJteyfKpu1+sVQJ+ecnIXkYr+UGOyeF1cY5cxdF8uHJQOGKfgpANzMkvd7BLpGY2hnBoujda7Tw0rHtCRzIqNXJm+l0YEc4yPbd4jVWN+x6m+UKXBV+K8ryGUjFai9N7bzaIdMqixHThGULLw3RLIe1TEBMbbXcM/m3flvybkc+1Q7W84JOdCbhS1xA9uB6ATYD8bIjODXea5h3J4oXRZGUCgjo3voD5WcfT0Z2N76Wp7yFyJouSMG4b+ns3dL92hQ4l71n8tK01zjP7woMAgAft5LapHkqM4xoqKp4dFhcPaz/GNaSVzloNFFAFwbwYX698dVPD52ur0JFSQ5bfVhq2fnU/pvWBC0YBHJxYKA7mLxZjyGpxoyXxZI0whEs6QW4WALJvlj6Ui7iJEHHTTW+K2pgl2qb6hG3U1ZjS679aEcSzyRVqEbTSWbOqRqXAOy8IRqU03BoS/tRzlkM3LbwG3kR24RlghamLaU0GC19N8M90BAtLpQ1EsE6irWF4k8YZAzG42pIaLe6hjLQ3xzisxhdv3gEQFFkXwmLNIUq3FUOlV4I34em5praKrKPM/JrSALvwg2pMiduYim8oM3oXLUVWrsFfd48wIlNnOjOmdtvUQcuBMqTpspxwYz9qJKJemD9Vd6q/bZP2QvjBfkJlJOSIjAkQJBTdvZCTUt52EqKBqjfaqxTsW/TgazT329x4ExqMUX+3wZ4Yjr0WpQ/LHuY2a1ZHt5UZ/+bA7+KS5gJswKoTMZkJ6TVxYERRLbzlBtY4sNMj+WK63qWOJT5JNzhxdb66u6a3bNcsiBQKbcqNevLvxhEYYLn1c1rgTk1uGZIOb9KkoNNIkMP8PaEwafUt5Cmso0co5FNq86E8upnNuTvz+LIBpUcD381dnzVNR99IjHu8q59Hx7/LySL1IrQfraVX5zu9Um0n5OXlsr87bd6rDodr3WwQaHR40W65jnjy1z04aLy4qq3zpfP52vR4vuCl/9oVmEUTLpVWU12xkXrFmVRaEHhi9kv5s4FM+Y3tVw4aI+95xFaLB0sADddR8Xw6BFMMXm4sb0vZ6w6Ccj1JspvYEngkw5eD9VQiDlFgIWf079NvXlQp8lysUcs29Q5NF5yahhdsjgqulo/kheFSGlT5ub4YYOZpPhcCr4ymgklLcdM6kuyHHwae/Z0NMtU4tqlx6gZXF1mYqH58m+irDPTtWS7jbv11YE4y+KMRWSthPUNGqTIlmySiK1r2DyGJ9dXndjX81RoLT14iGByACxg9fl78XdCPGwnp8DU8z/62GIFfQOOrRfDx959peHEsDym4ve/DkSREy16t4NyTmzb92j04lR6biC4NfOGK3ILWZJUkAvE/ZjW31ACRkQuCKNtUFMctbfjtgqkHBW223T8Ij94NSvn3PmUmmVIijVnlMleVPZrsLRi8s8IeNLjXbZ5xkOmjB3dkbH8763usD0JaUNuF4m6KMfsx833nTg4xCOliTLQqyVKWCOEWjPILk70SVMaDjofF8AmYEzjxgc2+NJqmJkuZlibwYAQZzUNqiZ1Dc/JeD0F7OxCNvnnZgMqD80QwXzcY2xy70QUu+zF/cKEvtwYibMVHoqxsVQDGF8/A4SmnFOG6D/VKVWvFsK2ptmgWPdC0jGBMv3r3BeDBxCLmQQ+W9rdRXBw6PEi0JX/520Mea6zatnhf0ojTSavySPNHpOisDVklNJw9rBZc/Ex5KixqHvVKS8agGEhk8fYZ/eBo8bJ7tu9YwBu6I+4EoNYoFbUFL4s7JnzYbc7MrJoP8wnSLDcU7Vs5ctvxrCi6FD1deyPD3Xw7sdcYqwr3Bdn5syX+2WiNJEAOgnS51Byz1awQR85oGef1iNHVH29PzACa9IbgPwjBLFPO8Jhh+WiwFcVbTNthmREisd2IgSSncwshwxlaTW/UUI9GCKPEA8NSi67zXYQcH1LDdv0j4ehho3ZuhfzDXV+L1cDinKVh48AxQ93u9OEbCE7Y4+2Mqx7C4pM9hvPmri5mYDOWaaReavsu552h6zP0DaX0lIRSxdSe2pQEh7Xd38DeoqbeYsL3SG68AYopSF3Pcy2fR4AEZqZYirhML7R1/RscZcxCi/don9VYu8swxsvkppi1d8FrdHjTxLRvbIhJeLe3KFj6yPouoUUShma0ZaicX8ASAxS6Ia5dy8lP5DK0TRFI9R8AXaTHlqW4do1YCvul+GgIE8mOm+vDvGPQ7me3U28ezKBBcRc4Jpdu8G9OCKuWXeObtnF7jya5+/hq0VSLZwBHGUJvTlqCAycZH9ijbC8U7XJuk5bbuCQ/rIL8ATDtUU4Tro22pemA6cJX6JRZYiocd1t6j6hslAt/bNoatgzNhRPdFiE05Y93OullpF1CujfD7U02uquUfqBfkBQKw3a1HRM3aiEB/Z5/8QgmIDO+sjI/lkBUTPAKX7uquN8blN4Na7gvlGbkvvY23csx8CNxsl+d3G1N4niYI59vUvDF4/caSsSnVotwdu1PIha1hOjsQe1YZU25KARG/OMy4w+o/8ir6BqXlB91hyjsIn/fzcJyN/S9xtMZmjeSjYco/T7Va9B9pSYRiB6wVdWyAHZ6TrnQI73aAYIs7ECG/5IQH3aYjaKm+WXmlnEV90S4sUCCoWh5dJRfB6GnfAk7QDK0d6h+HL6NWntkaHnt4C7z8hQIgNgdIQkMn0FGo2KtByHFYoMoz0aLD9+7zi7uw/huCFy2MYHk9md8nnW1mK2ERI1SDgFjcT8B5hKN+1bJCc5jP0WKahWK3W4wc3SC8VzS6Xy89A9AeOYKCaIARPaZPd/WOsHkTXaKgwChMGf7a34gUAgnZIie7RNgr07+8zapQ5h9C4iD9Ikh7fd+HvW75ji+gR+ejO2T3f79yoVbqG/tCofWzG27aEwc+O13k0nInNJO0z9gJGkdPZfwDEAlRYn4bi48CMIQDyL3miXjUu6XOpy9IJvc6tXcAMB+H3cHIwmQg8xETOVb0TsocSMsbVQHxIFcCYkGwmTToeCYSjfmMxe5XbDZn6XxH3Yzsbm4RDOma3xOFG8kfJ6DlSd8UUHu4jGhVsIBfEEOyOPDv3k2cZFmpgp2m6b82oXkFyklNqHgux8jjTSrwrsjVGMlDfttOCvNEe4+iY652n+zrN69oAw9+wqDnX2jx2zoWKFJTAztqu01XTFmHjGnBnpZCjt9Km5iyqqZVDmvIRN901TbhThJnn5+PrK9IVmctHQcqpWew2yEC3s5jKFtOC9zR8RS/Wjs59PF9DXOQFI8PCThIrWIhuyzmb1N2IgihT6IcL+fySQhztRn/Qdj9WEYVKq5i6tq1Tah3OVsWh+cPnm/kdeZr1lFRBjrUBhtaeFnfRvrYsDigJ7mDAImA0BS3ab7A1BQs6M9XBhcGi7BPfkN4CGPo98qyYcUkKExLUgeLsyuuud+wvKS3W8NE0wxyQNCCFEpV4ES/QNPiq8kvKdzZySGO7StlQRkdszMt1S+fhOTEkJ9P/RX40WyA8QMtjMBAJ+YbL611W5oV6aeiAuWa84/hVHiSwDYWEn9VF9PCHlw8wjFEGMxvEhW9/867/ULzObRqaUgbi6HiRFUtF7jR5p+/Xa+DND+vGk4O3VqOfS3+3P9Wa4tXVx7h9DJhspILxys9/+IVdcObJyWPVEe7ay7l4wn4NRKdS2PxaLnAbl60y0SKMxpplp5IIg+SxFjrT4Cy4Cjft4EF++n8HU4/1CU5Vo5j7iHuu5gDqa55gnY1wgurra/WO35F+OzC/dTYC33n8ZKmdf+9q9dBvpDtD/2N2BLdz4sSbqqw1kNdVjl166r+bOHPuNbSSgkpl5nwTE4rLMGEU29tvUTvi2M08jGo7Zkt1hZE/2JDmJONZkkyqMaWn2erNujDKnd9iF7z6lO1YtQZ1EG+k1kpK5O+rLOMYWgJBrMuAR+sirMfxCtUve67ukmo0QCj2O4yoXU+xRuKVj7SxGXxEJqeg4he2UR1MVLKDXVr1m2AcQl280Ma9vcKxlzz+BK95csZxzouMvYOtor3l1jQIIDX0+r2mM64QeoQouZskMGApq+CJnM8ABKnSCfCIHP2RSyFgveyxO1czE371k6W3gzYMke+EVXQ9IST8585WCB/3VAgw6L8dd7hFu0ZKYLxJ1HnFRORZ5bpNiDCbN1pKPevzMeiZUXoFeCMDuVWb/JRKEk1v/FI1ph1QVtgc8fnocg7F04woykcrM5bmFqRthm/QVoqY0O1j9LHV7c0xqo+ipVivoJHDf/OSUcKHowjnJQydWrpIw06thRyPONoirCB2aU+exwa4i62EruMoDBydieDk7OZ+I42Ps+VVG13I7PLoWK5FrmV+rrk8Y6+tNBNBNAjkHP5lpSK0yh1fcgvaHBsbi/VTJC/2GJcP+D+RGc7WOrBHmDM7fGG4x3IICl7EcVifPEL1lHCOee67rNrPnfwBQ6XyeOr7XTjZCEIEbS/VHtyt+tYIiDb/ykH06Ng+bD4e8Rz0ral6j8Dj2oufj1t0w9Skr9ihIES0wJ8Djw1q+nFDKhMeJOfen4bjAT/mvbtPWcIXXLXkTcRrk2HFF1V6cNOU/S/lL2kOOr4h1u8AKHw8oTuJ833iAKCXl09nTRemWKbvABzqF5kbtLwlcVvCn9zkmLqwi6fFmid6DobwcaY48dtiMG1mvlnXhBrKTf/f/4b6LX+TcefTiWyzGE5G5nhdrtTAbeyR9TtJzjU6mzay2ai36Or8e5CtFV1n44CPOiswLl/UtnCR7iXcm3jyRiOB2bCI2ra5UGVJXXVlg15nsEqI1uQTFm9Zqq8mnC7g7lFcsYVwry4qXD4hcAIwEHBRPq+r0DdI0l4L1pUH4f8+0xSBurl+cO3VG5sTprDYpZj1MMaoSlGIfE5c5iKTv8i8B2J8Q/4A2PkrFXxaw0E/iwk9l8B/3MEv7vSAHLEzUP8ANx5YT7rMJ1CNqIwWJUz3pinzMtglEYo/4vEVUKq/DJgmpZuwnuK9XuBJ3772PZGVh1pcPnp0cT27FPPsxSiAFn8PNdNfowYa0rR2BgX2Bmj1WPi/UVbATLE6Eit6KgP2NpuQDvoNAzvjUaU4+L0XhkCAWQoCLqH0+OzcBHZUjX0QzIZWKMrofdIm/Dp6QnLAsX3AEre+bjQBblZEi+PRRjA7XflFVWWyvf04jftv3AFivz6gCY3O5cV8UWPPGMzTbOgzAUYcxQmF87y/o8szaQ7bgjoR8qUtTDPwt/o2uo6UHmK/Wc4BRjgB2tDYyh6O8xNYvH7wqHdqTN+00aTEBIfFYIZGZ+/00rPpKButXY35t7+JawM6lw6ody9emDzeO+Xq0hvWY1KV95akemWbBzjlwsoH5noBHBqhPoBFeZ1mvI60bJU55pCkkiPAGuUSeQbozCOfr5lPbxWMufNMLUd80GMXZbNX3zJMTtXJSpw9Ij3iaMaJ7u4KJweHLSLMoSNrUImN5o/6g3Z/1PmHDdOF58Fmnxcy4Bquipf3RIAq0g91RpTDtH5JKiIsbpryn98FbXiZ5kXpozWoYlC9MzvcCBSgS1hFhhXwRSx55qyuI/S+kss8Vn1a1lLpREJAFHxw0GDhgDQwcvOfFevX93S971QTZ6mbWWtk51ZX9fKuMqXsSqyGT4XsGDT+v1IcTE6ByUL0gIR0QmQe/WZF9ECztIaGXFsvxr9/L1pvpfjeCbYNHgFPnKVyc0cntL3mEygRMBKLpO4n6qEPZ6/tpz6ThfMgZa7wz6Y2p3GqaHX98W3J6NPj4V1gw8jGLkYHry/nZ/H21B0qyfsXNCos05s9gMMT1OYKOxE8fZ3ZVnTfH9zmdr9k9LpXcAnaImY/jzUY3R86I4tz//QdI91IkzT26RwOpEv6DmLp3qMQB1i40bb10MZkbTfmucSPHuHHLS2jIsNRn0X8gygGDdNH5SG1jCBwRcbGVsMiPmj9aGiJE3DZjfqp5LFAjomNgZZ6GlSigYwA9OO+tGwE36bIMbuEdd2Yr6XcCIzRJvvrn5staakYympSS3+9Jn9bc1Vzi6zLoY/l02ABUv+ErlJijFWcYShfmLRssw84vXrG3S2eobl355RPwqFZKFAlDl68stV4UXE2xl64rm496SnHRsWb++5wDqBfAB1DH0P45DXVHfC+zAKk7CDudYoOBWJeCC3hPUiCGM+GNYXCGCfSs0kE2AtKUmz0LrBAnOBMbdD5/VormHo1QrtW4rwAtSlXFWxi8EeDDK/5uw6VBGE1/Te0GNZFHY9qo1UQpgnUz96ADSaavIQohAMC5z4INQ7gpqRKgwZSMZvpFMVXhBz7AyCY8snPsIh0HxLsZ9KCf46z1glDMtbiZZlMfEtljImOY8RAqO0rSyeGXf0nBZ6pJhkzlbiFbOoPM/xixg892SjG2uECJCHq+P+ZWTmYgnS1IrBLddnzEkKqM2b5NMyy4quhAAtfVre1ijNsRO9hPwmFXqFVIaLJbTfvfJIkosQIedCetx72xzNwPEte3NSdrswvUSgGJuXV6wpdaPHjmazSTAkLnxUCGdMrNMiIr0etPlta6xGjZY0Cuic2NyQqC1oPyuMcpgVdVtQyqosbzcyLQWheDzag25bA6FVCLMrxVB0FCVrdAj6TKhYM5zB72+1oCNh5c00ZhH2I4pT/p7LzeykBhOtDOcQrwm09GGxZPO7/ZJd/szXzliqzfNWtQnXPKW+SUCvHNIeHG2LfEF3TDiwKozAqgGcrx/SZDuJ/YcomwZSnAh0RDCSAgd2w3P3SoRjaJqBQ0XZxVU9Qp8u38qP8oSbBX+0wnXIaPu1M4TV7lyxH0h0lmV4Q7O3o7BXY9mGnipzu9Y3wUbnsKAoZinRCd8WLslc67HV4O+thOj8Oz6+0WOUarH1aenUSPfK9LAXR50X5tpFAK10VXMFnZnYm3gNNtuG/UP5uLviYvR9yY9PSkediHcSAbnW6pbIHMNysQbTbzPCfSkXm9iJQ9umUcpFeDUmjYZj44d+IMjYrvlLYYwOTBmG6az1lhryMmGK93ZtdEEjAT51oxmBhNOn+2FZnHSPNaqScw1NuWt4IGPRAVk2ekHOeD70SgtNPkraq0PhJAe7X5Lx2KCi6sLV+fD0fHzk+FcC6p2I6d5o0kL7MK8F0UYcAXMy6yQMuUBHggJuXSm+Ufb7gXEUODcqpwli0slMEIevJetSbf9grY/UJMsIbi7HCoGTZJy4J+F9BJCNWNkLzibF3B710BvcGTA0gAr3VpILvLE0Fu59vss6vScyKffNfSKhjjbZDUa22ZDssw+GkMTSU7LpApI6SBvgTcFvWOuXJ55iFOdUPWZs/HjAlWJUnkgui0D2lMHbYHMwbMppSJdm7J72IlCO+b2IoQIicQU4ZMi1K9uASY+5jzruxCrwjZi8H3hy4nxZUgs2UqP6mhItVZkf4ouoGiqxxNx8Vr+ILEthmHEAXuAVN/fIPwDr41EacuxobBnT7Flyo30VLPBnoROVqzfrZ8oqV9c7mCUjhDWdKCQDVg8qb6L9xX0VVXtZSVYegcfBLtxre9iiRG8nB5V0QHWZpnJmjZO4iXNb4FfswCS/xtaZPuSLchfV4Kif7xI5eZcMPPtAC72jzxxGed2vDRvwJlaNTSqNMG2vHifdUQnDUyVY9squN0EzYctALkAsY16S8uymqRJCmk2xSKM4gM4vF++ncobdINcntpxc6/bjD/J4S/zmUWMZuwI2badj+W8iahVhKT+qore9/55W8I+6gBjcgcLECqZFEgpGHNzUXtgRT2A97z7yVxM3Q10k1wHb+YgmL+qrltRQDKdhBeaE9FHp8/QXW/XiABcYcly4tq84tpx+LmBAAqtha1otnzYAzviN4EM0EyeSaeooOs04VfTnKOYWzZ6+GXCsg9ozmU1sUVoF01Utx6RHOmWOEJROf+9ZfTShWbr2wclcPXFjWGBfnJX7GjtoymgUWexNMAe6Y4V863rev94pRdVmYWFI7c6f5v9Dtblx6kjlcyd1/lBUryFm5xSDPjNbhNu4TCH6CeKXydt/mS+qzPmFmV6hbg5WKn3+sUEbcnf67PTS/pxV/TTtiNOu/H2AA8d13qYzBJ9X9Is9Y9uRQd7yVyGfzVJP65MlP1duBp45FK8DM3akgSNfHB7waW4qtl0lgFWFOvcl2LgLPcKgZ5juhy/fKHUE35zLpOIt1S8JiucFDYLZRKauodqyofi4hKcj9EhIHy8X8MSD0NY17yrY7safaHm6P9hPxB5c++/J9YnDpG4ngDk1DN69dMPP08Lb7CWzLtnOxb/V3jlqsl42gSFNvmkN2lbhbUL38f+bf8zc872SbxoQMTrWYVeFSrpEttM93+ZYhRyufCSRLpslXT1rWEyUYv26zYlOT8d2ih1nphtsayxHysxUBmpncNJkaYosrvgR31BwfVoGIrmoV1inTsxZ3z1J/jkKPKs7nCQNdGP9ew0xKAcgul0GYTLBxqM6P5jOIBPASInxEyLT8TqymJEcebidnA194RKm727I6HjSfl9410CEHGoVMb4oGyTF9vqKerhXNePY9p4ER6+SxvILf5okpXVC5mbVponSrBz1iZzVN9icQW+SrageYFKquaMeumgdtAOOzdwHrQZcnbqdrJ1N0DbWZsTDI4DUJti4Pt9v9Rhm1x85CSxi33z8wO4lpIp4cI40NjMEzA0voK7WAbXfDv2xPQrQIJo7luk8OX7n9nWY6ZaTt2eyrhX6VSAjYMzmjXjPO3oqwzbpEgiz6vtTPYrE18KLKIGPagd26UOmt78ssEDL32xvXoURm4sZNkSBlnoauCY6ychOjGq28EWE1GiW5xWt8I7i/QYp5oNaWBafJOeDiV2oowqN72ndrOJbW9CrLPBn38i/JBJzlFPkb4ovKfXIR30LYEIOXOKslszt4UZtRpUOu68Gm25Lt+8IXxyzvu7UUld73ceyifXBpvWHLaur5s3P+16I9EtZixVvxMkIoi4qaDXyazs7POeNpnztLVK4TiL84+akFHi/WlKM39jsb5jicoLnDOZ6ljZjO7FJ/OcwhPSuU+r/vYgR2oN1bN/KaInQZy5L09jy8vhRIwa5qvkWWAuP4Uzdk667sWmQAz++2S2W68O6SiYGEbs2NgQO4J/le7fRwCS3L6CPXrkGLq9T7VvLVClw91OnRq1TDX9iG+38c3Iw0nIbC+39i6fNuje+aXpOI8nYI5/s1mixEEJ04iDDN3iDtylIpImI+cSrYXNq8hojij58ivUsBJstTc+gwZZ9myPq1JwR7dkvk693ujr8iJqUtiAzdmHyn8+tuKITJqMVURsEFWJZHUHjrTb59kR99gBOyEBhi35IBHjKdHxbmzF/3FsE+HCRmzDJVYrGQ00nUUxkv6GUlMHkgxKHDtCSmpZlEndIImKvMYFQJoS57MVGxe9cU9LRBT3+ap+2cm885QOb5T5QEGNFqoEaHWDubvGMfvHgzHktrtWRotgbYVflXilX2ziExmYva7uB3P6b3K8Q0vBaA111YR5vTgNN26p+/ecD04i5orWqmZIwnx9ce2NRbPcZMS2f31Izy/hc6yoviv6WOgS95Cn3jAK70JQQOiYmoiwjbbO5tOIDr6SQfjUJ9xw3VTVyhgcaTgJecTVWvOMLoVq+PhPp1tPyjzd32310MX8EswvbWIMNkMLeVxp25Z/iDYrwjLbROrNHa4eQIivGg0QMRlYfJAn/L3oj1/UeUzsURxJAH5YgP+jjgLJuIg8uRtrGU2URlCzPeFzjhTSc1bIUsJcr0WnJ3arfBxT5af++ghYc1gFRTyvH77/69SkjQCsfMv2M+RYXCUEft2nIq/qt00VO1D++d+s+ahW43caIJElM6JXPFnuwAqbjO8qVs5vcNlKZqXNfRN0i6oR4gpT+DLs4q0QMr8IOtIbnKOkKygB16CQAJ5cLQWrqzruClkRl/w7Xop+PzBIN6goj6fShB+OAKQLrcl0ncQckur+P4yXaaiE0YjFOoFQsexWa9/TypF1sZoxApKJgius5ha6parMPTScfV1jfytCbz+t39kbBW7GWppmrT8mzsIw0U5Q6T3kZcaCCw+IGz3VmmfMo7C2FXftG91iBlvxeIHnwyM/lfQyL4ZhfqVICrFaqGOVmz89QokkQNakCijxHQb+RRS+gWNZaYE7bB7o1vFp9zWwMAWm74yynd2+P1pybm5Ybsy1rGjznrdYuiTVsNhX2Bzevldk8K5CshEJbYm7fPz2PDlNieq1kGcOp80xZC6iTJAp3aETLkLlbROe1FtD7n3FCe6smebXvH6s+A2ksGqxfVyC9USl24sq9A1R3avMLdfJbyLqza0t7RQ8+yqOhNxuQj8LFUEDEUz12Wv3fmh+2FaDSsLMCZW6G0/AVwxkpdof2YbhpEWNs281MQytr+haE3rQqCO91AzIp8nOX9qDZLTrlly0rbRRZazTJqZO6iCUk3HrnlbWlIgJd9Wski84/IBtJzyC9OGIOtJzTsILPGfSL7YfeJkqYmzR/E6nzGJnZ8A1Nd9BmzGKisF1e6Omo9AjDWeIPlGwqui10F6/zrQM65/bck4I6CDCgIPRJ0pYlDhDIcxbhPuu3BPdtF7EMqhSBthlvo2qvxSQLaWhyB/Si0/2bQpphtV5CUkz83zAwzJyxFdi3ZEgtkSt8FamXEu7rL19GKg5Jr+3eXMnR6Cr2x27lR6WpaevHvoJ5p7zm39Lk/sq97CCMf4I3oeeWF0FmRAzAmlrS3aihDB68mqkNs8uWWd4xPSjYOIl0J4RazLGSCrW4cUDLQGzwrgchjqi0wndzgLet63NrzMnEcehnj61yyapuu560eYUqbPqz4qIws0bG/UOmIpyv/szX2rOebGBVfTarmbELn9mArw/UWg8m+vaFDjaMaFNz5jVIcEhSRf8YqC5CVyvJ+NH/u+izDhJZjFFfBRcHOC3HwAVO6Pj/jlHKouaafUEB92fKNTNIjHcj5uYMKqiCj81CaugrpYpfGTwXZWOPY1eaDGgQ+s4kuA7RfsACTbXb3cKlgZZV2GbbAvCXuumSVInKuK/FIf2LAEheH/PEpU4hMlyoWMTl8JzA8ZSk13q5NMu63ngIFXQ+Xh15hJrNmTMpk1TtPQpzGYmUFiyuSZK8x5OIGGfWNXtJvZDewAXEC1ojzmMsCG0tvNiHYGLY8UATIxVW7CU/kderR0J2wAP34FtQRG1f1EToNeXLWVqftyXK/Wh/WxvH9wAirU/8yi5FQiyHXzdaYkr9Xal21P7oxBaDONqV0FTAhWH6Q3xJ6cx8Pg3LVmHImH5D57obgboiLhTA0ST0iGLFDQb7xWoF0J1Izef7uhFVm/nXOPM8XR3Tx8uDwJNa7c74BhtbvigIUUGO3uGRQdkgpKA10Ul7AblQE+JifJT3/dZrgAWvhnt88sZVkTd5ZRgbN5XMHEL7m8w9/w7tZqX6jNFuL/AFxIx9v5q74hKZddMAjRyJtvaOiAsGIcT6MQ2/kFCw/wykjilCnMWgFwAmEIhEgll5WSt0WXT5vS3T+lL04YcWcekah4y8uDAHDbwfPUKVbhbo3FOPIY+7IoXebaYO8b/a8GA3T0WTS5lL6lpWF7cqVjSpbA1sKM8/rMjKCgQHyUiCSzw55fFlmQadr6mZEuaPnemREjeIDjT0hqiIkjiyXlVOzPy0oTWixroQ7wjl/IcHw+tpBAA7YqQG2Na2xkPgV2nscHtmQ0k+bcFOSSrpI/FkKlatH+OxvETB/uG59zT/ssXF4BRbfhuv5UQ9Ja4zOMcGlmeUt3eyw0PWm+AUq/UP1FRd5EeRceh+tKhNXyurHSWES5uU8E7eFI+nPxZhO25YBqtl+Shk+CR3+lwcFcmbkB1SXcEyMOohqjIbtvXLa5bT4ThIhckGxxb227fJz35zs1oAUaZaY1tNS8pC6leC8fXRoNWpiLsWaYbckPIiU3BQeYCSqxeTn1476Fq28SDGnWuNjV4eq5Kglb5BiWIuPH2Vurpisg3iwFG615R+FzOBiDIRXnyBX9r/BpXFpTw2eVp1A8XCw81qUNoRp1GVy7KgBzh55tA6VEakSSuYGgSIz9VSdP1RGzkossR6QVko5gUJMhFydGn/3ofkdvzyVsORnExgxyVRRl5d2b60OhADA8ReDwl47iFLf6HGHkEgWvzWXohI0RcVLuhrHEE+E0iC5viRqLNRp5DqfZB56qaFCVfxBhJna1/H9ROkPxm5Ld2JhiMKjysOClqxfIjMO8aN1E0K9oHgsaK+KADq0Ilj58fG5AX/uMsm53elvLzhpfU7lN0ZM4JYKIrPk1CZlq29KjNrrYIAnF2iQgTK6jszC1YWgr7zT4h10KYZD4mfoldrQCpB8HFFe7CEHUts1Gty3zaaY8tPKe0blGypbXjA/chwxpLeDBdmpCD0Xm3i1EOKVV8BxDlgiYXBGuiRg+j674aRVUvE2ox98Y6a9uOsqrbELaoGXaQ76Lns+MJoe4d2JujXhIcXedN8ES/PZf4A6i5uVpOyHbki4aNIpI1UcXIepPMZFzFYuoWWTsgTKcwugNFvGrixt9hEwmnpqk2bv6IivwOrrawWyK0cIMFGGW+JPVdWpDOIXIO43OlSzixYZtD/AWXq3+BEltLX0UQ4wa9//6cF4tns+Bs8OSP4n9WB+sgNmZ+E2QyBcJLATFj54sP+3FgGQwfS3bH69vNnPfhO5AYsPHBm3jO/Wp0p5iBM9aLITgon/JEz8qfZoCpNTZtl/GtdiXCFC8baskd/NRSMzxp1yGRgtajqHJ6JsCa5VcMMu3stHptLKKjK7swl29YArwlPRnavNuOLTAGzO7bzn0T+EEVoEx+o7kOCuiQ7070NdpabOLfz5elHHjrDbZ0XTyZ/rD5FntvZSBFOqPgwOllRpYKv+41/YcQ6uHEKgGOWbMoBxAtZ27mN0Y3m04ofMOnTR5eumZOsXf3pDTDUYIXrrH64gH2QrmTaQIrFRyZNUvxNe4IjICqlI8uIkKXqG68EirsoY7px3atXfEkpuVDgGaUNavG9E5QZAf5UDGSVpFwiqA+u9MMTMC+rrPtuN6E74W3bTvHcdC4PFgVvXfCEY1B45f80ih0rdVjYMBmEsWHLIKvDQ116y9Ibx74QyfSSZeqkUl1fxycQvHwO+xsAW6mrmZWIlShtmOrUIMIui+jNGEbbQRNvjSafMZRU/Ny5TjFqdEMkUKcfpwIu+C1/2ydtKV44YF001LG0wNoa8pncrfwtW52FBVQQvOvJAh4Diru/M+FQAv7pqrMdjUKIUu7H6x/p0ZSWcMFzKqhYW849b3i5/JzJQJOsFgZWAls8EHkzFE9gpaVtY63fAhrV0MekqSXk6e3MR+cO8P3Ms9m2SDXsLZKAlSsGduuCLDsPppcDp7FhQc4zIv6feLE2MhUo8ngBqx8evSJJ18JYYLy8ZnA0sZHpVUqFPblM4u0TvI05JuAE7cLTqlNAtRaUErSjTsYPpGl7OKlLobzktLTgRsxBQBH00gxIZXNYeuw9/fTbwOA0hBzD+KsjIs0MZMdrJ6mo5UTXL+qZXToO2N0Q/uUnJtuFOQrH9BPMtwLxaUx6piByGTV+9MdkdbF6n6xUDkHH20ynDixy0JH6ShZM1imydW+VriL88W1u9YM8vNQJMevVAHjdOEBtr/nNxiGdsWOri+IcnDQtFB6Ny/YBV4RDKuxHTmoN8NGzbfdi77MnIHBgzu5ipFPaLnzwrOdeNNb8zkOE3NuBEVs4sFcCBsPbYOlszwdklyxkrEpO2QahlzR0dSxc1myeHAW9uKsrH+5Mcf0Lu77F0dJZRAzyu4gpAv8H/nocn1DaYYsAfulg8JU1u0w8uoyfb5zfHZWDyibP+P/CjEDvo+qYyBgGZTt7t2aBVa8bHA4OeUMb55xzfPE8jll5dIsO0jZcOZig/v8gXzlfLcxTOJ28rAnLzyYcPAsbyX+7L6ZO12RwnqsTjOvz2/UyfKzihYi5BkgdeKLNFJc1MdEf6EN5v2FNSABi7xMJktSvLshWFwQZrW3ts9VAHEG+gmjRzD/Zs0UtEKR8x3hw99InXHHEFlbzvYHzqctGqMU8ClILpYGUTPU3rTZdGv2HESibXxY1d92g4+DXRr3yhMaUkaKIyOvVBjene70kMz2rPafmkt8hG10hCZdMIBwQ2f4Hkeauic1DXkUerasAlIXnoGBdqEuNPur7Q16jDdO51YdX1lQ7P80UzhZj1J5lWj2YsIldehEzQPls7cdeu/KntmtOBrOSghd+SuHo7UeM1GFHEDvz1UhJsGDVBXhm9t4mM7yu8h5dO218cYBEkx8idCarHyUO8WqawgvmHqYVaFaTuAwizhs7w46aw4gFSoFXoSUr1nJQD/T44T1A/igzH2E+xmodyq1UfII1fyFnuFQ512eAdNWNpTX6aw2Cv0DvsK/ycrZ+SOPMescj5bYf5LUiOqQX+TxnzYBJcF/OyfnUdrULcciUrNsf0LmxPpUEyzDDC5Li8sd3ojcL8Jj7OMKUba4nptZm1Z01uNp4WhQ0XaaU+H910CLYoNPl6M9nT1/2gJrKHlNthq3unhSaTffRrTTpCSLuWsri+9qvrD80tOjAZGCjvAMy3R7QPwOfvMqyRNOozGWIsm/LOoPEObK8XKW4jVze0dWaBC+CXRAaUkin6t1/tnDysAae5a0dksbvc9Ift34VlflF1nk6TvPRVclDUUJh/Mj7GPJ/nW55A9HtTXvykuqFnFA/GnHKdhTj1JtKYb6V+Lkr80sLlqy/wa2zC6llRr4uxL9TwQ96QLIIvb8ZYeF5r9OD4+cuqrG+QM/sIU23AMmGyy7zx8lDWKE05SfTpN9iz/wjLCO+fmyE2iosjUKfNq5vWzmdZQAg4h6SmeDpNAVYoctT6xX7y+BGbbxfNwqL7FdQAANT2ExWY3kFjAAG0aorzAgAGE9yWscRn+wIAAAAABFla'))
assert hashlib.sha256(_v29_source).hexdigest() == 'bdb162dbc822622d34e9cdc0c4f9e91ee89fd7e11c33ad3113b4b6adccb5f3e1'
_v29 = _types.ModuleType('v35_frozen_v29')
_v29.__dict__['legacy'] = _v27
_v29_source = _v29_source.replace(b'import scripts.v27_notebook_core as legacy', b'')
exec(compile(_v29_source, 'frozen-v29', 'exec'), _v29.__dict__)
del _v29_source
_v30_source = lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4IdsKDBdABFoGYZjAex9znHhCdX19YWeU9j/POYt2DYePemSv+9QuF5cM9+qTQJKXBVdgyrWpAs9DT9cuKejOJ907f90GeMRpUDozRgrFWm5VGsEootOVjLNh+6GV6ancyt/hBvq/MpBGLeVEdAVyJVpESJncOEKsrz8QgGBxo7AF5moK7uxIeSHb0wm+KFNk9L4rGWfPTDQfl8n0h9WnArMkPy70w0acRu96JuUCKiMDHR0TdpDP64h1RZ4GoxfRy79khyQ9cazCR1B4nBh1FXt0c/uu6zifj5ghkJyIOunmgJDAGflOtJzAs2A8rq+amQmy6oCkV3qW/lqgfuGCm3nST5Q6fBRCxwjmn9VzzXPgflvMSiG0/eDTzPlT2uuW8nFVtRdPCMnrNv/2S34AslOSLyf8eKed9O2Mssf1U3ac9WslgZOHKRj30dcigGTncfW+BK17K8KYGnXTjQfBRyL7xjIwdQu2CR5v/q6sD7mhfDXGXbrzkvoBVn8VU8OoGx2HfrfvxkSS6zBCd0hw/kXCkr6Rma+IyxKfrWPOj/tz6AluSHhSgQ1v9c5VMH7FYYNx3zKiJ4ZhQlhL8bqPuzb93IMtvOHVeWBqnlHG2VQbnFBS4uFEBJPa1eNsZNOC8ip60WKUVxvmh0k61YCMbQAIlxvPttMHJ8JNT5iiKwkxtQRuhtSW8aDOixx9WiVVbnEYdlBjt1jkh6JVxFwsIZ2yx6fSTnz/S8dIumx76yLr5zFSgWlry/r+Vve6TzidSZQgU9H2Y+Gq7Ig8E3Im0hQ22PPXFOrxq5DPZ81pHQ5JT/tp9zZF68xnrItEbEnxv6lPN2JsoEO3XebduLIxW68TQARrBXgMigAUUIDfDpFTAhNGN9uFQOYveV/D2m8gC4rXQnRd8ImOH55pV8zH7yjr02ZHkyj3JngcGlhQQtGJcGEPP+HXclXOZBM2ui+K1SYBJHde5j+SqvNRAJVJc8UXLtz5376RM9QOtYAuUuRkl/220TAQzk2rdDExPWjISMF0edUnCw/ATa1XAiLTfR2cQuV3X/cM1OeaxOmwoghfbXErBMB0hXhXKoQJHYl5z+/KMI4zT7kjPRzqwwzcWUh4BIuFN8Y9tXGZHggEp0tThJqr84qEuk8elihkW5daBwePPub6IHJfDVdRGSsuPMv9pKVMz3RYqIjUE1raAzf0q7AhRQ/aFXpJNp32RsM7nXwXEUme9554jPSAAQp4AKA+cyKGzqbOc3EIbqbsJ59fiJRxMG2sq+rrtUxA8STZ0I4h6Mf5dMes1PqonEwuE1IgjlAHzC/DHkuAW/bakx52RzqVyEI+Zvq+lI+hJXdLnar83lZB546QE1DS49S41XsISbgWcZygktRjffL3csqZZ6PJuBPyf8pXL2/RmSTmKtKu8msp8tixr4E+qijJejm2qVi68u8zybCQl6AQQKHy022mTxGqIq5Y79VPDdKkWHCTN51SBrM7q/MfIIa4dmnyS+H5MZxJ4R1EEVrrCzpzDIPK6jaEub1s8uhvy7jgbnZS+yKGWnb40RnhTnnb3F3TazhBNdgFJKsbpKZZdnKhGDANTR7cJdup1qHBgsGBgoEiwF7Y522rNEmYYfQMClOhGiuCbYEkA38kaGPm1Aiuit6iG3VqA6KsCpj8G3ax2azfHZFvFuJRZKS/oj/NHpy/pnwUaM0s4CAzV25EY2mE0MArREzxwGdq/uRsesk49ipEF5ctvJmU/Qj8WEp9QPfmFPGxN+S5aqbhXuCahKmafLAUq36SGhCoVxoKUOnNyMasWmK5wYcZOQ7V4K31LuUJINLBoflTCuQRh6YrtZdp8QdVg44PW9lpZJd57bBA3POZbJspXyaGibgczE8ig+6VChFZuJHSb40qlt3XfTeCrsjsyqspNDjRNoCQbb1D4BE9XjFY60XGgfIPc0xv09ssPd51461mYQVKjNtQDWfSHwQD4l/GGa6QP1wyFCxVJ/5uNeDsvu6PrfC9dCLL0/JWbehGcptiPxNFojOLdomiUZJxr9BGtoFDAGGcOAcie1RIZUlC9NzSHApbL6SAFzJ2zIMHEOVJZphW0qyLsOKQNNJinEG+oNbZ0L4+ZKODZH/luBd7VxQEfiGg3kwMkfvgt3NC6dcrBbsBlpNSzW0ZuI7W3lavc88Ky+FMQHA5HvSwMXOBt2DURWW2+j+EGr/S02T6pA92scDXRt/vkCIK9Bp3AHlo0I5kHaUGsAPJc1axXLPgG1cvD7ErOglRymde+H+Rc/83UrbnX3qwRRGTmO9hiczUMghyhgsN7ws4HtzebW93bitaCtO7MbgfNb08KozLXN5f2Ye5mKMbqKWANwbCmwxcJayGH2p/o+Y2ZNRZxtwsBl7v3Egoew1E3M84KlCovr4z8nfOa/j0x2h0CcxMH8hZexFmaFK2d3p2/szCCo7Umlo1MHUCkCjcjXrbunLctVFHFDDr0ETVVYTYEpaw/Lk8ZOxJIPg/Z3EzGh0IeDJ1IgVdeKGA6SV3cigzHPWoMaENGD4VTfpasr4G3lFEdp6/R/tT5L4O296+ZGq6R3aOeKoFvXeO4TLnkZw3+Tq4YSmk0iNMognJk3gabLsue9wdPl5NTSnKOxSTDydM7LHiiAvYAaHSx13ABXphrhnUDd/1vlPfOKN7lyxJAHD2RtjCw1H2fqvfgYDSHwbMSGNl3wfs2SR/ZTDUVqc7UA74leDwFsF5+ISugyL7TETlN1viJKAH+6sfHjvLfoCdJ7J8IvsCf6st8R3WT87Fq+cwiCVfoWemenb8giWo3xG40V2hN7uoT9HpPAi7oSlTx7ZmoWpnrHrWDmd0W4mMZrVQPWpyN79vLPNbkCi3JHtEaldtbn6Q79Ia1IeRAT+gCMfihtsDsOvYJPqd/RCC+2mZuuML7SM3l5A5hw3pgA1fJKDtnLzdkAGyQ6hUVwXgbp3CHgwvcWOuYZO94q4AiKsWBcwC0A8NKPRqfsl/zE9j/c1KjDdd51PVx4leNHSfwGWHBv4VFlBlaRpsx8OAfdBdAfPUMfgI407igBn8ZHp66voZarKwctuzYn40+NMVZOHT+0KjcLWbDLX8CMKQMUktfhCSfI76CaUB0KzCp5DnAkjlnxQRds3dh+rLWQa97wTzieCT3GkFUcfjLHa9uFsQD0fDNZ9n4JvPDU/ZS5GyRYrxfeWOUdhnMbI8783tStEzpiAEBBmFAd1c7o0VfUONroslHHKkDndIc9sHQ6Kkdx0tMnwGKGihkdjGynznQqlX8D/yd2q+UaohBqnwV9U6L8MsjfDGqukeKc24ohjdmoEIrJA58U1f39OUuwGsi1iqE11ZHAc8bNSPu6rM18b4HrD9Gx1JkJ6MUGVlIDAYHNaqYWeAOElCzvOyvjHBpnNG3F6ngvMIOSNrBv+KyQPaHSKC+eu7EtLkl4vOhIqlpBkvpcVRRN+48nG3SMGZ0D/syGL1FLqBgPxFJ9TXwO8Argaq1C/r86MYVdkWKGKuQikDMDUtpzqDM2fc5T/C8ykkbj3YgMkp+v5hCtJgm/KRL2zy8943/ff+cREOoBxm/cgTdZXZzWzgY0nbPvDXL28gYHe/Kx3ux6/ptwMg2zefslr36zLnR3/EB57PfxETM+Rr7LvHeCq/xLDlPFeiXXa9YY+YYnRHhIA5m+gJwpGEe+4pMmfAYr5lz/9k/iSuXdApdzUnSeKaiCV3NKtZDg6oxgsphYJ3lk5G1iBbpffYSs2bb+fGc8WmaK8Q5joj9+STUbaVfFrHjTdRmUHmW/pjX/lNXsNifvR8NdDuKpsSHQk1TgEvRX6aICoBn3RZBUwRlQIHUSJeiPyFAL6fVqei/b2XLOhQ5YDQpF5okpw/+R5KSuWsKi8ItPDl2rOhIcWghbiaC2JZXh6Y32P/jhuqctyseOPGwSBN01DljDbcdESXdRi7BeML7J+i06+RgJrmrXexKJ9b3B6Hm5KMJ4RHRB7NNP1O00qPfoqTJ8eX4BnmWB1Zp5BaEldb10xPwwC9cBZluLRCqzPEw/G2FZJdkFm6/rJCUF6NXysRu75pibOFj0p/pb7KmdwytGqC8GM9O3MAMmEFYyV31LgycTEUh+MLgaKz3EFupMegnFrMHVLHztjb0mEb/jQNJjFTMBJwZqLbDN5JdjOUwujUioFc+Hn+zzZEbCvKK0gXp6nftcSHy4++wPLlNTWhVGjgM9XYfedgsiXPxe+d2JHXH0F5ksGyXfG7tqjVCVmXNhWoksUNak0xhXAmmjOJvJxn0DdpV3qjYx3pLxyXhvAmbaDFX/TSbbXnXCgBpfAL3wfeV2b2s10Xn9noApRJmavS2uJNCtOaurmxoYhYrLsT96r1iA8a4leoXu8RixqnLUoH3c8cpyJuQpFOEKB1NkuKeKSyw9QpYli2kr2I5eNbTUmkuxkH7rPDM7f3yElHwmbbAXnQ4TPMFWysb2JHVfmkdIvf3daVmmLspd+Y+gYIel4K67EIjvOGEbOPFtPOSIVqE9L5nEowV347rVfIyK1y+GeIzspt0tznXoXT/yddGAu3tnv22Iws1Zd6krc/Lt3iWPmJBOGxe0vm41mG6jx09/Wv7piv8HO7LidrH1pSYOrp7lKilnNccO2Mrw//n8kNVG8kMSBm1eMLcryxpn4Cpkr0ZkRhp7Ek7Bc3TRN/HdoWODgA38GaSU+8FpGewYQuN0Tpn4uFwng++FdJFoT1z4nNYQW20KzA+4bCV0P+Zh7/rgVvIy1BFj369OQkRfDB4rKoCTLnUsJNgJnC7tS3iEWJHi65g+40FuY46snHdlwbZ12IaT2JFi98S+sGaHya2mkNAH78c5lqoN+tnBLVg6U6Bz49c86upqst09piVXZL1dSCT6p/alriTTvp5YPoUshQUXUIuNkgDVWf03y6rX/cekKaoID8XtCm4pB0IQJ6gSMTq+wqJIKXA8wVfYA9NZ1DhhqcREgFtmkpR6geATnymeUtki5lpddftWwnb55wufM6NCBQmFVpGvidYhv9kskb4IthbRP7j0X7WMavOhg+058bOMbN1WtrV+mJ4eEFrjvMMy2f1+vNsXGPFZAi3fO+XLpEmfqStC952UP9kTkt4wDvyade738EyZRNIXvTy4dvHhdEKRzOcRf8nehJPXGq1YhjwEjppkHaAMtj8WpEk2AUsiumE1+keMr2/e0QcyZU8Fzd5uLy+Ba5ovjEY/UNPIv/UuWPeLvf6WyLy3v0KpuT4qR86VEEB9GPeImNKM76MMxs/XUOdj1xCEF9bq464Z3yyklAPfxlkjFfBOK4jW9AqxkLtkVUoFNZFPLJ2Q9/THSPUpKxU48T6957MEB4IkkNKw0fpjHe1gQkRlbHMfDfbHjggzlCytiDta0NWogD9geL50ZTPgSJw1TmtGl9iNANDaZhBeEt6fRSo7Q5lPi0FWiDZ+zFk4u7Pahg4A79MUPAo/HfxBz+cqeQE/IeOZOTlUHd3fEFJ8jhL3JSiz/IYSujSDhysc8RhM+yR+e88eDIZBkRIXt81Lus1UfPQrjIeTe4DEMYa3K9QOOWCPn+58XrVQFJ2E9+vzYTQnOlff54BeczU+6NEQohfDMUHj4OCxRALxG3LKxQ3HekrUaZANQxZVHQvSkx0WwzrLbyorr7X6k1KUfoXiuWuHAj0NYrj8FKNkyDVyBWt/CJ+Ys18spO0DE/tvyvvmjYTueT47xP2DpRcWQ2S6WODHvSPnWc6cL8F0FdokWwzi5bdZVz/lDiSq8uLenUua4oZyMn4l2kUI2elsl82CoHicGXkpVlR1IRYSNIb2JY7Vugyai13cTj1BhXD3GUol+jGUTPdR2AcyOVJVKev7gIW7zWI3K3UzUK+7gRgedIK5NJPfQlmnZr3R18ZHG7TXvDRX5g1CZW2JCN06Nxz9SGBbY5awSTquURfnrzHB7Pp5R14pIzNimGE/Wd2Zeyxmi9lLoYX/CHUI4NnkljCLyQxrXvQxDnhVDxMmSurJZQdtRwp7PwT4sxumz5T+kQRMSgoglH0HqqDeNEg6tEBdWdA5KrbdHi4/qmfvqxQHrLNCWgenhAV/Lk30Cawef8AYVWCOO+xRq79eMNYzSsUfVw0eY1NZaccKkcrrLGSUum8Sp8dhtjdCkFiUCQuDTU9SVWkfMdncKzZNVVtuq+yMHs0AXSPDWYvOPo052HE3sTJDGDu9DIxbGwiDWNY0N1HXavZpVAUbxIfvMY0H7QbNP23SAGSv/UsDi+tgAVGKxT3C/CEJ05i5IvRJDZXt8wD/7p5GiBmhXgyQBA2D2hJmf2iatMW/Wrl5IV8ktGMG47hoH9+A1YJRP6OPyk5n4QpCr1qSxiZZXsOOB4xYgPwucbMdT0PuErh0UbQgg7JxKQJSCBPvbAQIeRToopxNc6ZCjw25gyRT8IgWCBGhjWsQOdW2kudNJNtmJWA9pRgBlacYp+A6aW6kn9G96UohqQxBN+Mp3Is5/v4Hh76GWOns28KhkvldpSjWkylylA91jJLoPsbtao/f/0mzqHt+McV3Z2zqpXfeqZaiwiGUtVc++Udi98wMdxsliFDUMPEfDwgzh83KUcQCKThZRznJCYsqWXOr+FbBSobAYAUIYnltOnnJYVRtj4nuMAygEpAmP9c8jiV6Ef/6pgkORbmdiCLL9PE0nXDYHl0n5sjPlIk1tIozeUfvGSyFOLmPqBrfWGZXJ7kNVZkIqL8vYA8XvSnqJxnQDo6qIIjbpi3fh6f7Jyn7r035eERG7oyQNIbyC835akwbvIyQdLzs618dHXceV+jZG0sbWR5+wXVBvFyXVTiRiSI2zDELfYJRUh9cORYHeI+FpdVaXOdFIMEcJ2d/XcLYnjpZbvMRRGMZDcv3/AjpZCpMvUCo7k91HcASqSbCfiUSSWi4D8gltvoHrlYujNmUIjVHU6D0IIF5J524HdyQbyiAkiufOr13gUeb30OkW4Yh2SmxXpoMS9cJxt2Osm3CPwYYU3wQlNgw4u7YGweV6ZvvfMRvW1iaqINfxbaFNHheUzNznOG4FHPB8xWUZNAHIv1v+CWqBvZruE3SJFTgy4YvaMfwHGDaMvucwEAiyD8Uat3Z2Rekdby0V//tzSSVAfYAe2F0yAmLX7WyB+1cbF4TQaTTHpv1EdtZ73PG15cM5BjgRk8mu5MaXehUMT8Ro8wdr81AxrkzY6aRs0qLR2YQxWFTVZdaySSqp5SmbQAyvxzJyUxI1xBWZy/eNYpkcHX7qHxj+qLMEmf645ejFSjvkJVhF0ZxKF6A1VG9iCjybt7RC58/G2zqsEhHG6E0HAZ5b0heZPDMYB3TbWRqg7WDT+aOaGryZzICNOjZrGAbzzhwwAnJcsO6MgfSQd5VXbA9XkXXvHs071HQrMoqxK0O50JsaFBqT6Wf/LzdNGvDvxrucuYvWOuqWZhlT/NPNGWBV4PRllCfqMCIHTIECEtrp5Pso+PM2GoTfjJ+VsHoc9UZkUS7I1DNgo3NiDz1GOzf3BHOUklV27cvZP05oztj238+JpbwDixa6vPmDjY+eXwwv/IQcRkHyBhOea5mU6wbLa19iFX05vr6PPZ0gpf7XeiRVCrAGfk+wtcjyKNCpVR1LnP/Plnroz0UE2R75CEabOk5GhvBF7OdTip/6TKORk8nrj/GE+jXUch9yEYtuZm6Kz3sciw9xAb3sv2gkr95fTIayn8hHfNg9uGQcW62f9u31t/jehgDj/q1Mnn49U0vqBOSAOOKYg9vUHgxI6szRBHlL7QT5jX6IdseM3LHu0OamjI4azcV1Hs6E+7PqunRdZK4D5VBf+ApiMCnpmPR0CgYHGr7DDvKordHnOtoWFww5brpwPS60YhNc9ofJL5g2bxtABksJ65f6x5BpZ7CisFMgYtWt0zrYSza+hKQJG5ovAUPAOX0DEkVHHQhHNCc8QHcLaN5a0g8VNBMB9q8lWsqYK3NrY+HC6cZMfB6B5UGqUIVRHaGrvr2CcVA/BL4mL01DcX2CEuDYDJBVOkXAY6Uzm6zoSWoYTqstw88mNQcZPO08jXejBZAhCABguogWi0po2BukR/Ux5Q2+YBY1HQOfCj1JiNp5B/JCUwrezCiYsCS8MrQlSy2JnsruEFgqmUA4+NF1l8K9in6KTNjY/nOuAp9wolyh1PbwTAQE4/D3jvPCeDywhwkioLj0IndFeGxptfKKI/zZhe0VBhvu+qcpqnLwlLyTmttggoiDbf1HKFTiHf+v0tD9rOj0cpKyz9gFAsTJX1bKfB/50ooYBAVyZHma15mjECZJftlaQKMBh/s3syXGz2Wpu9u9lJiZ4T3aTivW/ae58h/J/2qz89Nv5KnxohPmDClH5sHIhAQMQw43fPDnQk9zhBvS0XNsEMRS1K4wM+C1q4OKm3Lr3lNtJFDpoXpiDkN5IL50qomyMtELZ+BVeEVuq8gMDdtBqV/r76eIAHDsGE60Pry+N6t3pXl4a2KyVn3yz2atkKwn/TqrmUNX9xrn5mDMNEcpJ5986rFDGe5S1AqDncOGHLQ190PIwp4HZWlu7ynbiDLvRW729RNPEO4hJkAxsVpAvRuEboHS0r+qlweVUyMb9AJeXoH2zSUOsPlHSZHjgHUmzRs3GNjyjNDitXk8QTtYXVGB/UdBInNuhpRprOKAtgSKIxwyeR/MRyNZauzCWB7NerYKlXCKhjTDU3mj8fNBT82r4cNC62TjqKiQxTs09N7QtEE6SRIw+awCeglyeCO7AMo4ZOY7mpNee4VBmFBo5SrKun6jU2BFEKHjp2zAgz4uGC/Q9rKez6imIdOJT4OZCN6cQBtqzLvioi7FdTDEZA8H2Q7UopHGj/2vRPBAbLhJItWBq+VzelCwb3RKBKuZ9E2R8ehtoUdjKN7ajcdpeX00qIlZ8TQh3pCSiT4ilMQT/NK07E+7EiHiZPVV3beauw3Xh/UxKQNK4VlRNXd37mFTtex1vU9r2Bz+PYqtfXJ9ReMzaweiTfguUhh3keIYXfj5u9ZTRJ2CzJRCTku7uFBJhiwJIaOrjjGsMudCh46Lj/o51UxgOmJkMmOup/3Bb5G2HEC5KxqcgvXrio6milJwEL427ppG9zAMf3Fc86bTYrQz1HKEi6s4R1kbowrkBtKLOgxj7RrAu0H1PdBBH3h2zRLzEWUQUPdKotHd9Zyj+C6AGzO4mmIL49E+uTxu52fq7fSz2IaZfG83YIXcIazyZnBgNFTh5KF5zYoCvfGpOtxtv95Q2SKP8tQ1W2h/S5jq8dlTy4OZ2Vb55NF/S1dA0L1+OgfiFiPQbhkZf+4maO82UO4loy71eD3PTcKb6RktRy/fBpb2y9sild1QVLCsbLnSIJ2VnEFAJv3a5dlcwHqsCvMKqJiYui4aUEyMyHVj9RCuGDMylxZ+v/h4kSdrftUuzGAKSHXblo5AssyqxKDpmHbefxJB9mqoE7iaH50oDH0JfYvRV7kV4zuaj1NK37swUm9VsWD5YWoP5E8LywpYejdyQKjyg+8QP7MLuB2vqZrp3CfkzeUwAiqloMN1HWKqKhPyCOGyEl8MH68DmoE8RZZA+ZgJmIGLJsRCivDuPfwCMrLRjiBzNnAeqkoUzBh5A5UI0Q+dII53yoqIzGuBiLWG0GR2bkHgp76IsNkAJq0/VZYR8CLC/onUudBR2wRivnKdFTxnfk+D/uedKsd2/QPDVI1UlG5BNahhEwmhykcb6nrvelvpygMXLkb86ZPYEdXjwsNgHgJjU4CGoLKPzzwAyxkVPdnDV3lGgRlSe4JnaerUqsAkcQa5MLX3917ClFP4C/CAbbuC9NqERIWaN/Gn3/Qpphq6OodcssQBNzsFGqpaGBcBgxkMpyIQjU2ucl2rJWF+Tb3/YVdvnzAMKdZu70CVMESBL32Q6gFxVaQYVOYBkLJSAaQmYbZlT8kXV12t+0O3IuDBwrdqwxzlV1DuobsqohO1jt9R4Art+6CVhaQ0LuLOFaU65q10A+z1C0AJq+Sd03T/VFo5TyXLR9u8HSMQ9UzicWOvoPHaRu8Dw26JrmSG5+EWDVRDWe5oyyGNoebG7NwQ1hz5vw+4M34GurJs5qKR+gFk0U236auOA9US4NBwgoBbLA+l5q/Bnoohcyenc0RthLOaMpNfzHWts2xMxTtYXktUdcLson5GluOf+azqwyScf9jJpu+ZtQXssTt8/j0t8035j2ojvbnzq4ePX7aU5I+FQLvzzPS5zMPJeUDosKWByr9gxgwEdmzBntvEt4wPYAwKGMbUfSvyrYDvfRaZvtl0R1+9tNt6Uz7yp4lnMLihjJmPntBwVtLppr+XCbx588Sqv8E/0VCpu794c195uS2HTmeSV5WSM2C2WXx6kXfEHenL5PN7vom4/rpdg15ZIVd/ctVy9O1kgELDUa+5LSakfJifP+GtLPybwPfnpZamzNBe7RBUKqUOWh6ksbM5x/t4mfBtTMSCkOCsZq8m/U79qnotFBeHEyKTepb9SlYGQsr6XfDeyyO5ahWHK9bhw47a205lqrYz5/oGoKr4VoivgRs6Yg7bQYxYlsW+31ZCWqt9qIDnqwOJ0mrOAZVFUnmv08nU0uIcmVTZfgQQ8GCHTr0+QSX3bATGYlBhAyY0gmpfhG7Y6qMa/9Z9V6Ut3Xcf5vX7OYg2xRb1llEGLJD/TtzJJcQFtAXpr1rvSp+fTK+xBueJf6nf+md49dkEHydSB6xKW8yMvyfeXB2fa71f+PBTeuAWCi6QyIRqLAVr8ZD8KrNQacK9oiMz17GIv1l4WTY/j+Gxr2PQw0ShQ4Xvc2Ni7iz6jL6fd0OoXyPSVdnJ17lfngDfhhlRMLKZguUvArtnaU9fzTFhOQquIAzUW6gI10GwKaOcOmEf0p9p+dAhTrHv5qltrhONeG52orM3nPe9cN5HIW/ZifRmLhvHjkk/Djk124qnp+k/ENu2dUVmWL6uJ32xxr7nfoafBN8f7aWv5uO8X/2L2Pm5ptL9caYrKHUPWu3BDfFWhEa/3j+6LxMpB7WIwkUbcQulQIZmmtzz3ozYaB1Wcv2BoRiAG1SJdLdOpVthh5ftvur15HzOAeQOH9mIONp0gQFCZ9ax3/4kjhQSQiHagam8+FolP8TUNROwAgEh01+R8EL4Fi3n893ZjZ2gKYXxb8oIt7TG0Cds+LfdNELKiFaRK5qwex4wo3ZklSGzol1ge0QKZcqZJWSTMKaSlnI0nVgahBD/5Ufj6+YOqoF5lJy5wvVAWoicW/vx0Yo151sHTc/HY7/vWePgj9Qg/F05Dz3SpHV9D+LruSKKDJXFrnzoH2EQwt8Jw3X70/PdiCSKIPF1jjXG+UgpL9qYWVpy7mLRF1x5c8P8ivcjQyE2+QZZUvl3Q0I5cswGD6uHYQNVVKMB5PEh286/vA2yoyrPL6XTp8JDKqrR4gTlebJJVAcG4JbJssZbqUyegUGQK3rpz8FQNREefP77U1oPp8M7cQCn+IeyqkOP7hJqicUUqXJwzDGPg7Fv1DSaZ7msEnBd4T5MtHkO1VgdIx98h1rvzMLETEPNnDddexFwW3I5t21Uaq9hG1HJUPAkNFU0TybrmblXLX0ZYGCA9AxuQap3dI4EeNtRzLKB80DSbs1l/+CjzsxjoeDb4KAWUtcw2LPICJ0mFmG66RM/mKETO9g/hdMdI8f5S8KHM6wdNwTcPhqeKikjEmxz0O427ih1f+yH9+YIrZI/nbQLQ6k2YodzsVHC71JpoN4O/U3nshUIppDE9Arn1royWN9occLXr4s/YL9mtRUM9Yceu8OLLef76CFUxPCPMYxSqTauVrbsaDIdxZg+gHZwcmflAnGje3oR5wN0sqVAXdzvC4vtH4YvCykIzia3UxtyRdvRUNMT2040f5ctndM76msq5RsRtNAdQZuCXKTUzfnITNr2wF47DDkhxGEGethSDKJ4dlU7/iDloaMmm+bi0wop72JNdcyimJHztlUu8RdvANHLOgqpPh6vvonGLnriWzCs/qDpJeDQmKWHgDOuiVKQjUM/R7sBZWMIMq6JOE004+5IvcqNpKIwoWAVjOB58oSlFGvr3epU+3Mb/OZiFZkW4SXeVXgMPJivK6xFb5gT63z96W+D893LgXTV5i3xJNoBPvtpFa1f8JYAkvcClC9mYfFngmRBv43TtyaFiS2/lhqm03xZFP55oSFCp8/e/V34koz+HxIjv/TE60UpOCB//w8sbCmp5YMKx+zmy5zvafAw+gyzN7eVLrgKShH7gUyVOgFMkBvshjovFOHCeQBOrQBRT4frmBKv5yR2M/CZfp2fG2wXq58YxD/IPg4T7OPClGAA4EzEw/gmf8IBYyVZDSuNOTbtNii0LBNQJ14rdxkv4FWWAYq7IHKz1lWgGfYht8WJADVr18HiE5Kyg+M0Uxd4zFaOeA0cN4wKYpzcoceDLMERe1CEovjCb36BAarw7DqYKVEIA4/KlSclLFQBK/CtwOZ7Q6sZfExdPMtV3igdAMzNzxoaJ2JEa31nl3Uk2owaB/uNua/t784SuJYhz4XkAjUVuxbWaGCjt/6dtezDiOerEBch3Wm3EB0+cu8hbPeS6d5PSVt4ZX0hqwMVakkXU1lwBnieHwx9fssL53Gl5olWMp8NzJMwB5WyexVTN3qilmlKSpLUlMnxPVcGCGeV/aEGtWipxl3NNG+qSRaxM0OTGXZTVKhME2IeYkhjALKjKDpcCVVdr1+0nC8EKQtu6yHnTOf3HsbXjnzV5FdDW18Ck/GP6FlyJYs9gqpclAvdcBPgrHPGNxa4CwRZsX8kCIVMkgzSWBfVOQxZc9UPGnnG7VRPA8JIQ3agbBUnpxn84N6kYSgPMlKlJYnma+nsHNP/v5dKqfOO/UPVJeDtJq+Znspfz4ancWSPFsmBM6D4cJf7ckl09k9A08vLn40OFKgEF5vo3o5/WSKIJ5gA1eYHerGeG+Ex09jMOvZ8sEJePprj4qQoQuC9SzApviXsIRdpeTd62ZB21fVIWoSAFgwM2Qv8lrD647LeuQjWpVpoSvkZrN3FkdVnf5P7/rox6DXTZb6g4VkBD1mtIe2n5DizlRXKvv+OaR89NkEpSsIXxKCMy8ACKfJCnekYuxNH4VE/EL8w/RCIZeRWcLuzLFzkSEY0/xuUO9sltbEhQ/oXvppUUUCWitOV0C4LkZmXl2ToZE9gUmVfgZTW/8TN5JGaKKUk6qI1PtPgmMkxwmWJPsJgi9HkMLnup2m+KUwaYSjyvC6hZBxDTeQt5c6HCg3cUHaLruoQB3XjPHNur3JVKfxDOLdguz2cR1vN4+aMF7Iiy0G6fFD/mnWBlOC1Zfn/18qQzLsPL9DsKC0z5vY2tiaIa59pXNty24iSvLCi4tNBOsI1SrM/8m7uVOliGiwq6LuUs5Iheja/LKTQ4LxfqeeIvQySUuSR7GdwZk+xlEp2tlGSUAkUrwkGrKvW4jv5SY3+Uh4ukmgNBZkMU2u3dYvpCI2gu4qFjHcpoy1UR276EBrZlept0rY8ON0OSCEF73HpCI10GvN+RZswFb93OLld5QcYGJO/M06s8WYTrlZPY1TA/cesWa4C3X/QUQRN12Kg9nfOvQY6QoV1zH9QE7qVNOe5eK7Wxug1tJFv1mA4NrkY/fzAlOwGJ+gWeu4xD2OP0ig3UmgPUZUAh1Vh68oQXoYcn5xMJZVJDU5cxbEMGrtU7tdMu72f9AlwFbf7FE2mQKnF8TkRszFoVvkA60tFlyEAZVMAAcxQ7Y4CADQI5GCxxGf7AgAAAAAEWVo='))
assert hashlib.sha256(_v30_source).hexdigest() == 'faf8f4d9b106097a5de7f11a156f4ea99cee809815970f2d802ae7f72634bc03'
_v30 = _types.ModuleType('v35_frozen_v30')
_v30.__dict__['previous'] = _v29
_v30_source = _v30_source.replace(b'import scripts.v29_notebook_core as previous', b'')
exec(compile(_v30_source, 'frozen-v30', 'exec'), _v30.__dict__)
del _v30_source
_v31_source = lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4G6OIqxdABFoGYZjEex91hR/0AAtCMKdYinHxr0tsEsewL1lmLJ8oBM70mCAbuBIaoik/y3o1od5NRFzzinqYLGzxM6qMDDYZ3U6HAQLhh5WdTRhrjHzXzyo8bJADxiILBszsPCwYSiKYRmRe3tq5baelkFxYW6ZovtLJkogBXXuJTIJw56zgL0q95K60kxJAZKT+cf6KJ5mmC1zY5g2YvyWjm6T0G5Db7MB6ZLPcRJrpmohJehsrF0UKvI0rUgxr9ImO+f/j0+ZjgjlBo+LwZT5HGmC/6BQpfHHJJOS0s3Xm1eQfbno61gORYKoGtukjB5js2i1Tl7iz5Gw8/1eepUcyLipZikdGUkjYfjQngjLCIovRwhkJNaU12k5Fjmbboh7wIZV2yq4Qb/JwXcxR8J+B9WLVLTAgZixfY6mZ3eRgvq1b+eECeSQiiSinNq2FsR7Q/zSoVLQ7CtbWpE2RkeXumA1lxTSGBEIVIoqGA17ERfYwPSfP6+XL22VhGNCftsUih86s3T6iavV6gstMSme0FPRe9zmYImIBYJPPR41Q6LlqjcWX+i1GRAA2v63fO0lw82ip9i4YnDeGyh6zcLwPnZlKL5wqpdYJh6vwJ3x5t4hSqQFfoyg3hdgkoNgWZvM13cHPuNoArWL+dFi7pSAQ01OGvI2AreYBtPWGjD7FgJn85iqLJFP7M8jwfZ5G2ZhaRcVBS2Rg9vn1GcbZS8KR+iTlVuaIUx0BCrqgLeHNAabGp4bzxfG70rE2WcHd2A9pGtfUR0oq69CpZoT1/Yeb/1Ta2qpBk9+2M78ewn0HheBLmsU2lZSMshVD6T2/BRnVJ/L/hk4oBYNDxFiw3RMBh9XfMmsLM0oGsEjUfeMlz+PToZGvuAFQBgC+5bpBOBuEjVr/VvyBey2hF1kf9fKkQe94lJ8OdtjSBMeMnTX8pkpDNDOdSdgZQ6NGMKoxQ48VPus1uSqaktGVqKTp3vMFE7UCk0FBG8apE9osv6fHvmpxC2bAQImNeKcFB5p2g79yZZTp6K1WNUC7sZrtLQPp0nQRtuV2125ZdacWace/hk1HdhfjDf3pbMsLIzbwIRQSgauRKrZNw+Y8g5WhxgaCHIyNqzULvGwu9GekmQ+LhIwnpIjpp89pKs70Hmp4ZVeL7yPrlVMMV4Wi/3q5lFHRpBA8QKVC3tCSBfmWrL5CsqFTNc1OCvYuMHmRjfoeJSh7RImjES1VZ2ie54lCxflCdqvkFH9pW9ae/Zz00tF14Zvi7tedtE7FbXSwE4yuPVT0DeFPTH33+hWhM9MTnWmwEG9U7KL4ZyGPp2iNnThRPFPXW6WHx0b5kbqu6yBfEJish6MN4YkH9yqPHFy/2hwmtnEXqhmI/arVC+uMED3RLCS0l9RJShdY1y7yBaprU2X74clY/E2vQmYxyooFZpyQuctniMBvGpBeCnq//3+v7Ejrz1qQIcDDUe1Jy8P1LRxjo7XXUHJy7JJBklFKeiefi47IL13p9eRlTbHgtK/X6FDBxWUs0gHZEvjBz55NX0CkCoLhEZ6Csg1jTPxzV+2QlXuOvuTsxggJZXP2cbjEnALKgn/eoIWPNlJh8TGC8wvprhudP3LvPQOYcOZvfvahsgh0RcNaxPvE85DCeCKoCXjms6UW5yrsjaYfSYqJnZt5DkSHBxyTqnP1fHLtkqwU/dwSqFTFT7AjTbXUdxaFczgC3UM1o0nUPXDtYpH7Vb+5fjmeQgUpYP8SwAh+HGJzjbPTgP9MSX9Mzj6XNtJzSm1L68r6u41avpBTEB3uBL+SalYJy5GkpRPWY0czNSBGi8yWU/CJCFAA0eaMLjTbQ6FaCMBNecCcCchyCuRywANNP8ElKvzSq0KLIyGUqf99/eEldfQhvy8S186gssj5S0vu6hISdCeVByOp2QD2hIJWh8QbfWodLlKHg/iP9MYO9uIoFKsNLaIPJXAxYon/5TQsU8zyOeUj6fxGy9b0ocRgZAtSsXUfdRJXFL80RuwsY0e/ldDFQBLTPm5Mz01XumfIzVvl+wUKHP/HOcaU8e2QbJGm8Byv/4HWPDq9//tKW2UccUfIV1FEX93lp8JuX+nmKCNmktKZhgTJ4SkPphf7muMxyDa57poLUI87b37CaUjJ6eeP18Syd1iMKeQLNycfF4Po7ABWVlznMuNOPPWkBT1cfHyxkz6vOLd+Od+00JJV7fcdpDr12/PDeg6UoA+ga8deLpRgMpx35zlSDuUUTZZmGHtykhChPEULUsZfhoJlGPsNiDy0Oln3bnZW3MKb1uNN1kjeoKuOfzU7tMxXDS1hlUcXn6cpb6FJh7cndCtnkPCUtYYVE6osOjrg/DUvE7dtyqimlCjX+87tU3Scx51mWmfNRHkcVfMX2drBqkoBc6jdZ67lDJkfuKF36kAhSxxxAjNCtMRe+DXK2O+5FcL38Yt/ph0UPxrLQjCtd32sJCX4X/7rLL3AkwAWEHHOGDqFkaEpbF4A2LYq8Fkk+IJcey53/P8/LjHKl9vNIaCWK4sF/gHXC//Dh/dG+23IFE+uT4kjPekaiRfpYjD0/KaGdM4MVd0Be8OMRyQafw4L1mP6CHLpGIiDfiwMr2qkrxsHfrJ/JKVZxP7mSH9WNJYgFKG8CyxCfNz/7VV7N157tk/0+NjslR1m6QxcC3bvriPH4wnz1zWd+hp9nxJUeF7+aBAKbWmwPUwFB1XzwI6Sln4Bb84KR40om9jBIwDgoYdGzM4aWzprhiYDwe7T6CUnoSDALAIptqNIbJWbX5cG0SHHm/I3Ff8rsjK9a4YiowSAxvWJv3VJDK3iMCVArHNEzCEqu3w+UcKnNvr9vmMu5hvKkJrwvxjL0bU1ZT/6hiop15P29YHs4Q+O4mgxcG7CaPBzUx7zRrzarIivM9ecfnG7S14xtoLK5ndJlGs8A99wtwo+8XiP6ZXO1VKXuAFHpqHkInv11oMntVzqVkb8abd7Ob6y9FdHOKd0FoKgGVKtbvbj7UqU0kTxsUqKIyHgooL3Xl7Ow14wt/+IL0J0lIDQmgWjw8yXMApBFh38c30SV70NBR5vHmGXhi9UcGCNTHHGUFtZ5e0oI32hkCHUZ9YJvpt/p898IHP/e/j21ZFFSDQmHDXcHzeELJMT/yk8TY7eTHMXEAHxifbpBUHaYFwvuZiLs1N9SmlURC/SzoAM4cM4sjPGgnbPv7fiy/JhD/YGoayxLCsL6hZlc3pvZvm+caXfQyzPnjVeNbDk2kzIQImKt0pIyIEYfwO1miIx7g14NeK3T1zaqmYiiWIlYiW2eL+bL6p6Oc9y9x1mreE7OaPfVRahdKj0yVW7JIl+B3RYNGVpRnTfDhWpd92fGNThqhuU32UyB8F0Y4XQfKvT5FGVguBdDOYi/fpSJuWzoCO28ypCijlccG8aNJWNmhBO5ipu51YytwbMbyS/rNJPds68LWgh33R6WwKaQn7xjKnXQ0fUyRS7QYItav4JF8drE0Z6uhcMMAknv/z1ZT9r6mXbyPbiiBxhdA8B4NDEsLIc3r4Jx/DJmVUh2mXBuB1X+QA4WoMBqi1N44pTzOVtXkjiqxrOxpJQu4CZEabLKNxSPNYx640Eb33Wrizc7SzdxjFkwoEg5jn427qGx5NqCMO+bkB4l6BY3pzg3l+6ra1O4lwtYRcaiRXoNaEDE15L67qQPOsPXyvI5EHuAu+UY4Q3/J72MbC65dQ7O+O1XHIer7xQNqcURaNAi4s6byqS9qZ7uwi0B0AGGHCitlo1tGrAP5fev0ORqxvYJ9mb/tAF3scMR/aRk44PvHCare4lK0M4W96bbMm3IW/7Hz+gIpBe/YGjwQ+76JwBm0yRnLx5eaQvLyIpIqzMBpIUSRyl/W8miesPhgSK68/YJ+5ftEl4O9TCVyHD62/Z3tERiqeQExA2evNSotb7ok/slTAaxjVG9GVIqc15/KN9jHKyTVVh4TMo9UNoCFgyJXagMF5JPivbMBBlLksuMMOaFPrVw6mhnZq+f3XjdyeeSeQiVEIIPUCD2Tz+Xc5/OdkDjGpALL2VQHZv77bmcBi4+OJy04HrEBExfurw0SKFbkcA9R5Zi+2uobkdduQvltO2p49Shfc53/oFfzdALKt382kbgCFtJtsdnuLzfkGO80fXyC3VR8Bd0iw6klN52C8h+qVPxDIvXHFJUvp25Mt8ug2Wh2U4qWJFaJQv1Kcz/w2oNHJ32LbbWaPEDoKLEoEW2sk2HoxkniKclzlsig/fzTrm9gTDU+8AMul9m4HPmbxpdmazCbfby96h8OHJ+LITo6GlYuPGswAXrdc944FxTPQpxy0/K6PQZ2xkhKvQ9otFZWfWDN/bdJeNAaV1x3QkOsCbRK907CLaGMxhfmLDGVaN0Q+Q6d3OLFy3kC5lwy31hZODVBfxLt+skwBBqPLfkMFlRH+h7ECM0PeP1kQAKXbzvBE6MfHp5sszlY1h7Cz0yrjcbyim5Jb7Wf+Gyfftk/aqZpuVJ18MOzYzvYN+bMsbWcNvlZ0AO66FsfuQfmluN3XV6y42QlomQYqciGoHnZz1ludVbz+4dhgc5Pmf+PQI/dOO37dJVUQG8lYvGHPmEqQJCJ70J6ixmN5xGxEA/jB1w/QezzpWUmpX46pxllEUMofV6yea6bxRt9sHEvDYZVE+FxQkVcjkPqj4c3/jgUcX2TsIEMHbDKgfem8aCka3qXxNMYe3uv1Ig/zEPT+8abr/PWkUBUUD32XnhiQP5G1g7oBpTvV71rRK3c6w7QaHD1nSlYVgxpgFHO5V82GEQcO6gww/hffwzov/JZq2Q+/VnQJ46+kdlvtjFd8t8EpoFnkq/QAdI4kTTJv67X5OvQBjyJHuChVdNb2Js8EblNIPE7QZajumDDgphS6+GpJ0akAZz2HjoHysqb+PA3kUdDBEOQkf2sXp1Jt/wjh32agHMsoMflmRtFqW4yGO0+mVmBTZtAB8OzKEw/weXZmCKB0ffBF+wI+abUeDrdoJATZ74OGJOi0IrjGnneyitO3dXq1Qi4pNkgrqnfm1jT5qzYgNWbTVXQfSFRsE9jnwOwjMP473eBsx2jWIWFdzoQ0l4y1w5PR7GclvHujqqsHO4j/e1IvkL7sYgU5oHY7ATne6v3AaYpLYmtrMAGOm2fNIxcj0inYG5S6ztE7fWO2PYN7Cc8vUIDcENIeoMmhfEPY9lhs2SAkh11SYojvdKaIAFUSQGe3wC8fLj3fwpntm+hivQOytdMCY82qdgKbg4valQ6250W3GFWBfkt+qfaXvNXyDlZU5WHGNSFwkwcbIC1gDW0nkhfcYaz6KA1rSvLxbcpBOcvPIuX8NAAJTy+qW4p6ko7EXTO1SRElu5JElvw4X3hCG5YyE3d9LDwKJP/1Ucn7qQ/A5B76RXRfNQXb4Vzo28MUc3ywVhwP0aLM9Q6XiVJl4hOX0IiqU/S1q2h1MSX/zgZ3Uid6j5KcwBlgPt4Yg+dga1ZlBACO2b9fTg4PtzwESIdhjrT4H1AZfDXdj6EYNYEO3SOk0sEPFQgd3LcVWPRjRtUNTzllqz6dJRIUJ/bbazgYXYm8RWXazEI236UDREWIHMITgphzkU9pifhgHhaJq2VhR4dM7F1pyXppUb0J/k6di7FdI/Kwn0jJHx/qSRZdMm8NMz7CcVEMmH3NstfP6MuJnbwWJCUUvnesAArv8lulA+vhkO4WjBCM7dZtIpyQdRxJ0tikIp34N902t+hX8rr57s3pB5PSHESGwGkQIJR3cz+JcxcD9Myd0sQ98LO+mTe2bwja8avq1c8MvGu7ivh0lD6g02X1qC1I3z0oEMLIekj4hdNSiSBhA1D4nanDWOMHYq2NZJ592T8HOCUUktCWTRMrfSv0oO3dOzsw9aPGgnuLRaZZeUgOYnI0onZEBnfK9mZYff6W4GvnHe3pQJSp2i9POxnuIfBX1sdtAhXAlY7ZWlD+oNzW+ju4I/IMgCiTBcEU4ykgefwIDdsOeelADIWh3UsMQFM23Lm+LXnzUJsB5wwRPMgnaax1EigG7V8HZP6wKK0oHfQGZmZZsuvy/9aLUPGUKIfpty+YxElKhmLjbpXL9HI040dcmPTZZC8xDw/X4ttL8NGBLhs04nsqV3GTNEm0GAJXmnUSUSdGrJsqmf3m9GQbtuhniWhiiaORx8L42v1eKiKHSyYFmZnm2cRspgUS7+qToDpAzT/kav5sOoS8nMjl8lfYKB8h+DimXlWXGi0LKQjlRCE5fyPnB3B0uyEvH/MsUx/dEU3pynZ6uocMtPdWxcxXwm5TgGAd4yceMrthEFtj0SrKBdIMmjn1NW1CHHtisMhmXQZJn4S9qdQ5/Ri1s7habljxakX0eaOKJwDvb3Bdao8u1dYmsv/Jpmj3q1JjDAZtEbpssW8OyvjH4/D28Jb1Ug9RRbukv0UA5UviAe5spsmQJdRD/uJso6gEnM7+IuR/eNMsZ7Ii+UJQ92BtppKJ7xE1bf6QG0k5l9CeBGMVNrpwOoK+rBkGEiufaIK7fYfxEC4nYyEBFGIHcOURkxYYsjl7hOkEQFn89NQ2xsrh+j+4SKDAPjUC+tXBWEcGonZc+uNiATD9824ijQ81DP+XpHEPojjl7QMDaRXm33i427vBEZ++Lykl9djYuBwIWSbf+bAVDixJZzhVmP47son2C06CdhtlDxS1gR2kN7f/+DsjCNXa9WXih7lhan27eeHk98gvcp3OPnGiVsKr3bGww8cmzf5vwDG63sgcGAKNDG3LAUmbjyA1ACzop0R/QK2HqSXxqjUX5q5Vgccpf2uAHhBSyDAYbeTbi1zFk9w+7eb2LWVFelJMsqClXummpSDwsIb7OaV+NXn1NTis+tGw44wBHS3eihnitsPhhbIpAOCtCJ1im1lMuHl8IhXFhOLw1/6vxi9EGn55D+E5k7bQno1C/2t8R8GSf/XejfdG3IDpL+C5vtX+waAZD7W5Rcz6z5k9ZwuyFtCC9qDI4yqd4638cAmE9zcpvkGo5KmcrHd+b43vAxgmujjvudq0zFt9Z3MdkzL+IsknWnX+WUgxORYIyQvKhBgY6kBV8VTsvqqCaHj1U5HoYNjf55MBczmi602wW8hDM8Ka2fFgZw8ZDxwQ9Yqlgma1ofyeUzq3vnO/JCwMuERTs+5pEo/UpvZOTN6Hf2oq7Vt0xDlIF3YV0iBmsHZw2ydo2+ooNK+O0LpfwigvsvTQLg/Vk9m/aia8G1iTfk4Rf9DMdQlZiEycM8ctAm9GVA3QTTlCL14MT7TUNXhOsITNdMdVBVZH8sLC5nQPs3ryKAD0ATfrSegNkZKpNZUs2UZekEZo5xo/goM+xO1QFiKrKlpjlPZScTr4bdgXmVzQpmsAdA82CfNnxryxcOk9M4cTfNEUXz+2zx4jDdiQR8d3R3+3u+pimhJCcNY2AEJjN5Y/15gtXrtfXP05vr/PUC36gSF0ypy3mm8hgpOeQi2dcR/5GdJ9E+iPWEYFHxKm16RdngqSK9NwwzKZLfe1N95b20xVcfmMy7tmO0TVu4klKu4jBUeOtazKjXrWNK8w7X2hA94SHsKadVDLeL3/oG6SdVEk/C7ByzBzYgLsGg6Q6OjCeDTfxBevrnplo/496ZYNlkupCUEemvzJL42Jpm+sGY733cfP5LBAe1ICYQZBNxC4Q2xirD9uIkq6Jv53lwk9n5levTKc5tp1IdYeZUGFHJytH21qlMZJjBGevZxoqn1g6PAgPFP3c13Lv6KX2F3zq4bgHxvQ0kUTRR0fAvuCEucmj2n2F8vX0g1iMR+qGb5/wJ5ATWuarnQ7MtwGgSTomdR+BCdLEqMlGhznPvLdZuCrVE6/2tlCFT658cKetvw2vAAIeEvz9FqZ9RDEdpD4GWd59XoH2qzLEvwFB4vOquRHMy2VC6qrgbzVM6h3B8UXCRZLdas0bAAfD1UsdjVIOCbVVeVrFg2x6LcWrojrA2SEzAf2SKCFmDzMkAlAUDelhz9RXJIH1J0PWB3nAhjtJ4mA9kCrOX/FtAm3+CGLzEDAWOaOP9JIjGVIlVMYezgGxBtvJug9eHLdFIF/TZi1vYRUaS7yxwMwYTDSCH4yAk7OkTEKLVu9htGAf0cXPCpaRXkcyV007BBmnDibcL3zhYH3xooBTYgKszeV1MdUX/j5jU0QuweTennVQpej2KQr0Zz7QukF4ZDarg/WINwVpnnG1wtwImoLY714fJuTrfr7SO7OTgk2e0Gfyfihf7+yx4e9AsSOENtTOvIQa7V3bNcLj3HfSFhOA9fl7G3KF8ttpVjNAhZsxHzrzebsJg/SQSTayjejDxRezl3Gw/vlfc20OGEyuzvr4cTaqQEt+Xsl3QIqv8zLchLXOnSWxC0urYbtE+xf/RCVaBiPyM6EE2CUylZQJnnmHrBhU51dZVd7p65Ed2f+YvM+S1kmtDF2D0LANLgiEv3e2TsYAcbO3ZAEr1O+ARJWhMmxa71PAM/wypcgqVhtHu4ai6YCeM0HajWBfgalWImcowlr5aa3tJTfkmuBeQ75TYdeL0SZUnoXCpOojZYg8eoBDaq0PBYyKDucdgD6WK6mTj9/1xm7II5z/q3BA13uL5M9Z3jlwe0xI/UGj94DIjJrW2DlJ5c3rued5xeHI5EU9q+dS+HSYmtj2n+d9W6ssF3+tALEnilPn/JfZGz1W0/ijrNKyCSRYVqwEIElrYhq2QfDo3Zrn64HU/Z3gNR/HfR9Jh496/XVSlA8VsHWuhlJ9ppvrvAflcjSN+LfoBlsZu7EoEpmQA8Eu5+Ut9YE1I7MavMIEynkLWnyrjO6LAJnGuhmeOC5fa5mKGqYAyVAKDwo7HVdYN11b8kcTun6wj/L5vow7MELLXe5lTyvLZdMn2P+gzrmbTTGcr1O1v8zvN7T2AumObSF0Pd/8cOfAhIhCY+vMEw32NzF74GpaUqGeYJonEgMQVCxgY5A1Etxg3ebkLwk8TumpQImI9dGBKCKWXd04DRUR+F7rD7wv5LyJIpVMUXrlTsF/su47Z7PvYKxwj1rXAcGWF+oyMZ9kl+LB10kaLGaTKRZcC+Q9CDBwP8Q/6m1A6IFVZXtv4NVNxkx5nMCSvxhLVNzj1o5CCONKcxppEN5nWobaZvLt2+Ue75r4iegtTnCwXQvl6Z5eMnf66Mm5DigO0rK8OUAYhyTIypD6TkKeHm+RDGlsg10E/gtqWGhHaDrywrYsQvClOkOdhqk5suct3/kQrSANqaf2b8CQnT5114nObvtp98uaWgujPT1ip71G+PgFkI+0tcyuz0GNDg2kCOICd0yPniF7XQk6s9MK2bzJT5VTBRrGaDfRndUYrr6PSCz9Zsgqtjdx6Lfo13QY+01Ci4y0CX/T9EpS5IK0LZ5Yz+LEI5sWXReSVOMmc3UNWV+B3DlNYYAwu0gwzkKM9EsXr+ThMZBGg0CnY9PJkrKwpmCOKR+WrTEnJl++6j2ze16d510CxrdfVMl0bES28er0XOWewkwHPUTqjx0+S4r0tHf6Dbg0KhWFFeuvqXd8bWZO4EjrG0eB2Hc+6s5nCr94szTw7ivQ6wg+S5xDClokzE08oT0RT7jC4D6HmAdHXwU3sQM3p4huMjQhblpsEDwnGcJW67vFwC4qMMljCGQeV67kANS2IpDr2b3T89+y7TwslJWeGSfWddhwVRYaBUg+B9S/FH+HXt+mjFuaksw5IlUbuu1jNlRnkotcHYCcR3PZSjUF/pM/zzDY2AYh9QazkrQnnZdIk2lRV0DIHl/1rrVJHOItnqQ7RREFWbbE5dZyErgRZhGRuuDtXGdGk0nHX2mJkfWaGNfr/+vb11CCRF68RXEO9m/PTooC9i4q8N8KyQ1Q7az2cCrrEMLIZNKGdJPa1KO4TqvxPeq8G2weR+zeEDdxyx5m2Hf60A2dmd5w9vwj5Q8kw15ZnC+4BD/PCVqvYD8PsBomiC8LEBApGFRS1cmTzLZyRoeDu11pAFZ4KRAOlG62z6OVqUuxAnAvdquOoqHA9EDB9mfwWeG7JQ6NhQqTTiVVwxs3yaISXYH75erV9Xia7V9qYr2cWZyrCxxEmr9XX8G8Jt1JrLb4BRTAJe/hxIUzZL0bnGFJvkzuppOfw0asyE3KJHMm7bDU+cky3b55cleGntYC7SML/gjB3Z4NvpUyBh+56bdrCqSq2xhl83D/flI42MQj09dqDFwHG5tEmlcfCQcO/17Jep9vq84CjLLW01N7aftpJ87iVJQLtE9tOFAvD74vd7QmWopyuHvKwRBzpWwoCxI1TNWtcEKoVzDzoh+d38vLEcToPCCM/0ObS2zAsfrq+jSuCsu+gLqDpC6n7CU8Ygb5rSM+HLuv+jfD9IEFkVlStNlXGXsjELR2DbHuZ1LW9/zrZXgolPTYAE/41tFwynhocGAqA2zMHxg0uNioWR7ARf8ehQ03+tvoeW0tLDenPSML7GlN24oi4EHx0EmU+LmRz8DHYGJnWgRsyct1ZbILD/32w1DGzsnT6wYGqRkG8lcC4t9vCpGZeV5QvT4XG4FLtPJ6P3HuYSvAESTIHPZHyjhdZJ4qhyMKSkTik+8vvwL3In1zMUibWqyZL8g0Mt997VI1vh7/8FT2NqjJcFMznIvtp9njacNE9OgyRwnqyZqtnNrSrQLZP/L5f6WMOlB5iHOf1QawDslvALLMgIfxboFzJ3ax+OKLJojEKSGjK4enlQNhWNADIozd0gtOkDGvzs/BwMTie6G9Kb6TUB8cdrivIgMwB+CJeGWkwKTQTFoBcoUyG7bMJWeQUBCqQ0MkO5Fr4VZjtf7YP38N7/4//7rlr02vo2RGHrqLvLkrfUBtQC03EzBCFKCegOMRGqRsaPB9L3I4vmuu8rgOwLU7WpEgjoFO0PlDA3QKIPGMFNCCPjY5UqutGzlnpr/6A1Y4Ypg8WbQ/US6rMLAS7F8M1cpHI5mrOvd/D7OTY7gojagXwv0v7ynyq0c9oEsgsIBv6bgoFnN348zV7SjgSDdXQ6/O+zoh1mrpmjQcrUwGEO9oVSc6tsQ6521VSZYe4UDWpHHc6Dm8u8C1nudgaQk8BWQenylclR/p1pZWZ1GtyIF8mEqSVZb0tiolQIkEA6/qsBQOntLKL0rv5c8wTd7epXrNPaTXHMhqUtMPMhNNnfUCPG6cecC+aEhua+PmNOxWYTZCKKrgpedc0zsli4uSx3TNTZwg2bv21c67Ue9Gjy9Is9BfDyoKoXZ3mRr7dX2JjZzWPFdGjE8GhycLh5vJPXyAcmAzSP1uFs1SUc5YlzCZXgn200L+ieX3/TaDxbTjxOkHnLBULjEvcAa/vyg5Sb/Fox3i/DIWLqo9uTUepZ7X8GKHwrfep0QpjNv3UqQYGHt45LGgjvS5nx1ZRcoSM+sXJx5t/HJ25QpKn+zVbllmNUbeAdSee8E98VnyvB9CeEg12WZ2Pua22cuaxYUz7oj8sMoTZ+j36wxiqc9NvS+1YOhwi2Wrvj/p1lUp0Akv9KUrw9lfhDhIv3SRyF+T2WbKLHGowWzeDUYrRhyoDxa94k+HQjV2BHlejPNTyfjpNxztsWneyLnCDOl0lxj/9YXjZcQUK4lmpTrk/srPi9WTh4Rqzvuh8PsK2TQ+4zemBD5UKf7znfbE6zEKX605aMcpTHSv3PvyCVJrVJEwrvgc4hhrZ/PXAf2o+p9nmV1OxEOo6fqDUwoFBuPgCbSe5zst0fNw4JCMw4nKP044oyma27npr8bIiIxEItZ6mqchhY5zkKDx7EtF0j+KPHg8MtV5/JqQwCVw+nVmG2/EDROroP7Sh5YBqVL3G8il5ZsBizh6RTXedYA07MIn+FmZvtEXVAVRUYAAM3x1WbVnm7LAAHIRY/dAQByro2HscRn+wIAAAAABFla'))
assert hashlib.sha256(_v31_source).hexdigest() == '35b414a5a804ab2eae663ae619d564c5014094d081e01531cec6f781bf934fdb'
_v31 = _types.ModuleType('v35_frozen_v31')
_v31.__dict__['previous'] = _v30
_v31_source = _v31_source.replace(b'import scripts.v30_notebook_core as previous', b'')
exec(compile(_v31_source, 'frozen-v31', 'exec'), _v31.__dict__)
del _v31_source
_v32_source = lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4GgvIMJdABFoGYZjIex99zEAnl301sv2FfUjXMjKQAhYr1K54xV/Ut2zefnKld/6WPOmdDK9mMUAG+DuRILhhmNjO7NtQ6TD2thiDaB5JAh6hpoCbw2LTh9F821139DwKCjuDUim4P/rPhdH1MBmXgP1ZxiTvdp+UTqmIWP9WQF4oxuwOuNW5xTnDjY3yicrEMFWeG/nFbdTMMXgooBMj8GVuLmeoesHReySIA3dt0JwDUQQJ5KrlK1DVlGoejpisxS//GSBSpligxRV0CL6ziXTnusLqLSr8EsGxhb+dcGx0MO0IFpoqUETYd0k5jFdIfs3LscEd29VKmWwm+0piJUbTVt4k7l2yuRUmFCkH0OJxaG6RU/NM/DeMMX0dP1ZCrowHA6xsMHw+v4cNWchWAE/S9mexZ5ZQQ4I92kJ8Pj/5eUF7k9zq1Lk8V4S/6RI2YmdDy7H8nU51/0WnbJ+JVStdNRhvk7qZCRbNtPhJvjZwp7Eh1O9ecxgV2cu2w2X38yRlj+jgPBBVJKPZtqCJRRmpTezBNuGuK0u7dmif/wAyv371ephSx6HETZcUCWM2s5P/aR/dQxTKaYIFtMDFyremhzi11PIArMZ88Z11Up+FY3kJdi8E4/sFN+2NncdkGfppqQyeFUKbq4eiSW0ejNNWVA7uYKQMnLdiz7zZiSd8RpbWepy0ugBaGuEoz3riQtrfEkizCcVt/RajxpQCU4oCfI15lQIbIqaSkcWWIQCdTeFns4V1NUciEr8YHCXhJkJqc0ha1qg8ADouo0rDN/1GX8PkPsi0Yf6J4a0WtjsT3iiC06o4ibHNY4JjzRfgKVajxL4c8s1cc2X0Yikucy9b6AVKySmcQnQK/CyRkKhKxDp12wKhDD0f4yRXn2V1QiFZxt2JTHtJ2taDLzl1HJEMi8juHFu7YOCA3SZ3zbsDjmBIfiJ+twhjahjp2Eg4eDMKR1UZLfJnkAxz+Pp77TpG1/rcgjNB9/zdrxylygXiZiPsJqNdBi/7NqRCEu1ZdUd/5k/V3TImWtJz9JsprilxR5Tvl6u2A/Ioz45JzZ/1Y/IujW4N4I9xPGZzwmpD2R3CzqU33Jer5UjD9BaBAO5+1jAoVnJz+KLqpUP3mcnVb3Te8r4ZGMC1IWiy5SM1vhtDbkJLsKR7OG0fTzi3QMsKbatmYO1wOl17OpzffNw0z9Q9Kc5G4WuN/Gj8gLZAf75AOPJgOqSNpnN1dOy5nwTqyyAp/erHcZYVVweVY8tvxxQ22EqWT6+GqTL/WcUJeUxBHxtNdF2qweXIdy12ImZqTrqc0X7C6augqxhX57as5UxAOdrkHRJYi6ibsZqhVodNRTIIjDFmpSZLhUEUfGINUPdz3pwLD4NtGvjSHAfDEkbJ9Ey7HEYtkx440+0BaAqXWpGw/5AHdqnKEsKAp4IY2wmCFcVEzOVjbBwM7Kv1XZYOHKUOplt66yAY0uujmjQHnxSRC2OBlX1S0GE+ZrvT8P7FaOJNyMeD0UZpcenF2rXqrR2JpWaFM5MH3j3HGZy7uGof+kdDGZj4sZqz8KxpMhBpENjZgFQYwMQiFgPK69rlZ+2HPI4ix3qJCUfoICNaGXlaktPcnY2by4J/25FI80KNEpQDlf1a3r/JohBDu0EeiXW9ZHSdcvQiEv+nEm0JROZDre/dfzvxW0+ycsBJkogwr+plegPBHCEoCzuCZpIw7HiYzOewZyY9iwaAeBdGenJToGGivVX1tAxUPOhplzxd6bhR8Ii9FCKviitL3wW/rVyCpRPkQd6d7kZf0yooR1NOP/2TtQ4M79nCly8njZbyKLpZ+6F+2JBZQiPwr9ikwV++jwM+ZwEwqscKP52yPTkypDgx6FFOiaBl/8tArmRsTKTNWpmc3R+amcJL/jYiTGyWbmy83g0FcONfGrPnfWFmPhAp12O+QQg6ZpgIZJYvojKBRSUF7UrX0lh0K5rVMVJAewCx7Ymb3lN5TwdPKfhDsQzk071jX27Y2e+11a5qKvW4EMPUBV8/KHChWCSpzQySI5BsGesBsMmS/N2qI+myBPGdwBI4Fm/bnCtqYbEOFtUNA15oc293tdsaxvU0HCMAxbuTNu4VjjCqwA3rbSltJ+cDrG2Xom5LOhGq7tOf9WnlEsUKiVJYVM41sx/lAi+led9KOh3b9RTv6weLP04vXFutsflXWHR0dqj/q48Fvy8d0yuKdV+PuvYpKeXOEWoc3i4vlTOnl3f9VIhIgrAmD5/EVd3gRI8OhEhGpw457zxPB6otm64aW0fy93e+m1cq580LX7oIzOH0fKIz5q8DTYfKxDFzO6pLuR5kdtdh+JzMVXrVqzTuBZ5cWmk4JoKD7BQx47AphCk+y13m14HeMz4abNAjj5EF8rQCxHmg+FTbWjcytM3fvPbY+bg86hretGU0bqnY8qxOAXbqh3qgANrXmz+j1Ly5Y40pTjV495LoVtLFmlM1woOlgnYcYBbd1nyYGkGDRLRhOc3KBHn7yf8YUVe9N92i8xLRFNy706bNAkHGBQlZqBwuP46/RCvjCcH5CYuco5DTTXT27C+0olVtTtUE2UpMPLPAidV8CMYuu4tSCIILsRPyu5HKEXy4uCR4mn72FzghfzZtSBffwoOVPbDq4mra3ddU37RJ7ecjJmZwO5f0u7aZIl/7ZshOUxGq05bvuBfmqd405/MKqkGAXvkDG/We01SPU/Bgu5/lGnMG4p+UC5e67qmCej3f7hYScvySYauzPJ3GXKmrEOIIezsTTxQi6NE3fix2/wS+si44U0klFjeLk9F9OZZaayfMt7CHxsg4RKXlDStK8ux8lsy409Bcpb5GYPZCGlpUP/9KqAD1YImsaUomaGFACSqYC24KZg4Dcr/gWxFdnxQx0LVRT0YKDQ8D8GG7fAwAKe+kuZPqMNpXRCWiOIO17AwROJReHkpexvHettDmtM5a7sMOXSItciDqQp5Z42SQypCQrI5b1Si5UlzvrkC6jk6zZXmrLJm+ITtEeJ2TDYo1Qwaj+RPjS1TGiCBa51HxfpE2ym3Q3B4IyqKP/dn53J1jGkolVpwD4SdM0mb42KiEf40WXIYIuTGEYVBKvIizorTpNwd7XaXlmV/RsVpy/aO88abKgVhFaPEqvrlyr99FvA5WKDQe6b9WJZtRplFY72XU7nm5LYybAcVl8BIHsIFt0pjM34mxsuqv0NNh11wLeqEY4OJIlQrn1PaUQEftyFCOqbCYerbVhzxFi7V5TYgFN6Gsw1hXvIRpMbzWAafiljSg2pKRrv29wYOPcYhEFgkbXrUagXWmnUkw6PKSoOIQw2st6SlJ6whzJ077rVQkksQOS5l8FIqIGulisp5QsAtoEvWORQlGI6Q+mU1DZ+p15abSvNm+0tSJNTzrTUf3vnSS4f312knwGcvMx+ylMui7JCDVOBlcJ1obRRkeQU9ohtwVOrysI/mq6d6rdrPH3sN5eszNWVLzLkqSt5ywoBhhlJjri9uQZtjMajugIJMflBOwkWLFwkBt18wh6vkbbaRnql56oGGWXP4Mga4QMT81f4QrHFMs9uDM49y5uEIFDIQbd90XdxuojXXlmid5tVEYOc0CoCW/GP1gohpgeI13cTWBhwShoRbAk1T7Xt19pb8bT++OitjNscUqlSZjDC9Q3kRMFAhzOCKvXYfcOn/Pn8NwsQzATDDtEGZO/IXYEWz+vkDb+LUfNlfW6eokEFi3P/lea4I1NIvTnRRxMPSb4uDnOD0ZRMoLG+rEzacExWfRGL9SI+3xLsIs2g4j9iGWIn7wrhiLAB8HbMHhKdskcakEHbUHW+81JpDnk5AkrQ2eSrco/C32jE+Ud8GHXM3bonvMwrUF2x3ZeJ5UJJ9s1dWbAOBpCfAxEqt8XHXV6+2Ho3fJbmKWcC+t787o3n5E5dBJXm+cT6yE+97nd+JT28PjIUFOW3DXjUGhmCEMMoBx28SKFJ8pZ4RIvmeg8Jz/Q7UuFwZCw6VQ2f92nGAaJuscEM32F1u1Jt1jyNj2Oa1Gj6VGNfrtsalijeEnZR8zb8cud8U6ubxyuMREgTjpNncsZ6fBcotEmWu9d157mNe+8QQMidQnGEyhwTb+S2LWRIOjvSpheq0bxd8xntcst2UANzw1QNWeCrFvM1fsaNSpxtfhxwWJlKteZufIiMwKmnShb06uJpSjHexyEGDlcY0l/3E/ffdIEQ22njpKcnaww7vQFaQHiUt/8lKer4OOFwfsfVoYnkKSy/t1SAwlcFFduOrqRkvrUaDU/aZLMceSg50d13w/KmYDn9dedj/cOQrwWMT5yjJWTa9SoAbLBCeh/yfmS5Gk+WM13tkC66GwqWOtrGLYfC1OMPaTfaVd7SHJwCHG2oMa9v8e9Q4X+JOH7JjRlN+GWXV2LewGyPkwGxZxQJiMD26w22rGMXSLiKEh8OtsU3Tt4hzvQftmqYpmbriQ/3QowHhvoHvy62XoOp+Gs7c1JQL+5vRMydFqDZR/mzgszNjbx2AannOqlflMsATIhWyvLSabVIshYd5jMC6KibFpiZ9BUMXBlTo1OdcBT2pJQb6Y3Jb6TFKRhiDsMMfbiz9bgBY4q9AMjOmaeO1Z7F3dK7JELsX0PVBX02yiBX5CDE7MsNzquUEQfcl1pTeVHFVYhZMiwkVZnz4hXzD1vQHHgNgcNKAspGqUxfjArybfKuvSRHfO0+SdXS4X0gUf8XwQVafO54LSe9Rej1ib7EDW+H6xdpYtie6Z8WMPaHFhKYjoX4VWZb8jF96y6GqbScexdRoPyAff/mB9fMzu4iAoSThwdOeu7GAk/bgLuTPwQ6VXwfqQ0q7t84JVClxGTMQUdGlt5kz+VpnRhETSTATWAmK2msIqbJ8Zk5/SgwaHrCWBNjoKlKImpU1ki0n3UMH16fcphk6ZUN7lt44zj3jXMDlSkyGM1nnUDNB2gekxk2odZIIp6Rr+cf3niGFTQqX68tjF9H1R3MgRXNYgIoogR+pWHTV5QZgJdhKoJIwuNo1+Xl0hEEYk0nz0KI41PVhbtC32k/qX5aSqIkMonBEYoMEfq/IWoU7DLzqrfHEasc2MlpXngr69zqgIuYbjAOCiaiPOWBhp9OgnOWR2bNwb5izHMCkIyqShJHdkJcWDPFPpEVjcvMJOvcF7qvWvTTPsN+teXB42DDQjWSGjVzXRf3eg2ocJP15OWL5fEK9FDlZdkQzoJq2JDp90Y3NXUs6WPpsNPL/QkYCwW2mmRyuOtMvSXCk/k535hF1775ilodoqg5PkZ7CzDHIVnaIhoM0FyLr9BG6f1/9zsuSR8gAJBbGlasEwvywsKbBbMzncEUhE73lM6FruaG7wCGee5XJFUYtzGym7mLmRzRaP3g//GQu5vAGriEOzb/wntAceu/V6pcB2CserpPW2i4qla/TSjyReq6P5Cpt2GoF80LuddVtZ11G5Vmd+rsorsB5iZUI/htp8KnOwnJi6NJc7gSl2MkxxlPpXR2IOPMLLUxzDURdG4IDdydwk/3s1ohsgGAIqq6Lw3Rkh9Sf7W4QR4ACTEXizRBJJLh4PZ1Fv7rQ2Hq00caEtwSgs5JClITcTIBp4yTMjqOpLUP/inDe9Z+Rig/xLDEFTlnynF6Svm97JPsk4UUvFFxInxQINBWWw0ct/Iy+bACwgQ0ipCiNIE0HvC/fXlrPy8ICHpxNqHZ3Ar5CivnBx6N6NsSDmiC64YkStEMf+QYF4anVIUOV09RjLZSw/QKPQKd9W9Y8q40/iR/yY7EjO29tFZjTBEEnXLUKimTxFsXg3bAdJzrgDGoNqZSTbN4/SNDNazuQ8eMRdxivCIIC1GllV4jM4GE88btGjTfvNZeym/JfJd7UD+jGMzoeaj1qh5vl/6/NmdW34eSm+nMKP3HNKp8oLTjdoCzMAkFPZtGxbFWt1xz8zSReQ++u8hwkRDf6px63KOhJf39vlpUbdy8fRZzhBhjhl+9G71h6x2uDCu1an/RSzwkC+FuuZMITi9156MRzrx1ExEO1HedyClmZLf7sSVfGX114Nd0BdCsQnobSy9euvE3uA8quGWCm1YcGUtpXhESbGyXzzAOG6w4jVfKzxYiBZhFcOXy0BoT1ZsATzAJTybjTNWztHU1YXYdDFXNcVtXWkwYwlxshwvr/HwLb38XFmAZ7JQU1VXEiOPGLjc07efwxnz5OYLrvpHlClcU2wxTYaPbOTIGJJ7rK4OKQsA5lY3fyEo5xhVToKictuCShmwoeBLoy6dVWGhj+ysUHhsyLiuEp7/xamKUrUSobXo7/tbU++c6dsnPn4xNpfcAvrwbFYCYsNZgOdEco7nBZClLs1CC3JSQbtywzQCJCjxgbqkYc7npz+pn8gQCAZQ7fOdXQZ+Qa4TvP57CzCINK1mFjM5zwpnbqEFoDKUAUcU1EwqjbmlQrubyPWFQpCk7W0HnuwdbdSfN11iJ4mPv+Y7yp62cIV/toGERrb9lYrbpdwSATOGm+KWPuI/myrScBRto6/BsNG5t91/TXCcjxaZ3PHWesDfwKKY+bhwZE3+7Xn3gEw+hf9YZv14czNb6OHIn4AdNMbIFL/lBCs2j9r0lRmMZDjaH3lB481agkDF3lMr2euW+wbnqP0OZiC7ooY9jqIydfd7KBUKgD8mTLiGSEl9hgjyJ7M3MUR7ohcwdt0JZjQWrXGuz0ycv0QYAzxHRibZHfOUflxqeDi3AuKrnz65/cO6F0eB1TWAM8uso0aovHr45gfRiYoBvAeMUrOdoJTGnKnXJbOoxNNCrwgZPlZWYiwbzNtVaiVVW8hlAFePez+DxxVN6oIiGYaEqYqYYOB3y9w3AALL1+G2Zqw/Su9K+J/p5vR7wq7zpscoC3mI0kpJn1zf2lMYbFfBDEyVPfvKONSrYe8ejvrnCeqwAq0qXQWk5qZRbNewk8XgNWNb4Kf2S3/5a6QvQM4CBLIT6cpCsJ1bsvtTpA3G9UlYYH1cBFzYj46cL848ygaJnxyGxZ+NLiRHHiKcg0PQg4EYbbRKx9nicFM1qFtsHJ1W5kZduCW9pkwBCTlUw+TSEUHonm3XfUZW4AlzFskUtuzCPEfxCL9/Bb7PsXyuSExINLja+bFshhsX9n94H0nX/TQvoFSuca6rt/C5fLL0lAMGRSWXZnVuAhVmwo2R0f0o7T52tKkXgYsuyDnIz53j/W+/VzxZ/OQxi5+SUgEPW2sf2PpgWzdvnhBVN+rqiGDj/equwg5lQaRvYpjeFW42EbTLIJiZkeDfcDp1A+4MA1ETPwQpNXE5ONo0UE1ngoAN15DGfSHdH0udcQjsuMMl2aLzPTZbAeKjHItnMKf+tOuLws6il+YVzTDirhssfUmSIWnnebm8i4Us4sv7O3wlff2yHqU8EzWPaxXh0qcm6M9N6qKVkBQu8+BhqwTSQQ2bPYcG12/qEdPhF6Xy+Mm9A5wqxhTaVWv8UTPA+Zij5WDDGSOF5Rnz7OK1wrlPE0lCN3BPoZsgF3+xyGw98St19UCm0ra/cTq/+YlaulPBUDKCADuOjhtm6hoMZD3adUVVjJ0N1GfC8SDi9TLbCmM6+qgp/T0bd+5lLMZekyXO2gGPrp+0AgUzukFIKTtPF45IS7DQF0OUm4vB5UWMdRFEaFul+HBi3zqhvCyZjQdQedZSnaKR6TJME0F/TzfYovcLKIEBzwv3qwPdITm75zoh9m2cr/Xw1Ws8EuNVFVhRaTDY/TnDS56n7c5qFbXQ2MWmjU3g88LgEYEecmDly5J6d64ZfsnSlfpNzWAjwA0KdUI952ZeGvA7z637GVvEqVb29voaUhSXXPkW78Ilk+VS1el9jh19ik+NQoQDBa/y2ii++PWDaQ3Z4RKYlwgOsZ6NdMg/RP0FmNtp40tbH+SqhoIZGJM+UaxkEnGMhlyX1Fe08uULM5QBeK1CcWQKs7fIv829OV61INN8Moh97OZPWON8ojrna2dBbe+iHhAuZbAkIphw1jjs1T9WAp/gsIdiGFYZi5jZwjFPK3GlYrSLD+3PikdRPqA6ocS/Vf7WeV+kP3k3sECWDI1t/Pzdk70iNk8SI7SCwiHhJSmdZ2eb2He2GkyNZdzgm/lNGwHGQnUvNmyLFG55bgRqW6Fngm6XHuZ650eFGnsAxAYZ6sIN29/o8qspamRt1eaj5fOt9w/Afndsau/AeMcMPDIJedWw3r3oO5kJ7e0OYmSVfDu8C3pdxA97avLgraKh2Ktm6AAakV6nOwDkJkFDLqYdU2/03cbIR51tbQ9tQNdB1AkjnEt0rKvRgrxT6PIrAaiGxauZMIiiogJYH+NF+G3VfGHN0M+IyoRZJXhNkleS3nVc5AQ4ujvW0J1v4A6dojy/yyr8O5Olj7y49lvPWLBpPlN6bE7jX0u0c2vZP3qcakT3AUmWSkkPvozD/xzTQmBXnd8dH2Vqcx3MXTjw0dA80aeMCdDRBs8GEGIdzMT/96FRcm57OfLUv9x22R3B6pLmx3eBKS0ifFvXbiH+8WWiGwPqWKpWOI5RjPKuXimC1xSI4vgSo0oJXVn5dUN4a9WDWydU0LG0q7Iz+1Y/uVEkzFLkjqJn9QoTM/X9/rrpqDAygNCbYTmAhwN71qoYNAvW1YDiQA+ybE4d72nhRM8+8EaQtbn6cFoilbWG3ZAOmcX+dYZ65hwbCuEOwcJYbB1Cic3HNFQLC+W6wT8wAuVQKHcEs14dNdzCxL/KcoGpeRADjwyPM2wB/3zXXGVE1/u8IiSoft2oIxxOAf/bRwp8ARWjoosgDIawxcSJMw7DwGTwDz4zgtSbHI9DtP8O5zAW5GLDjFPDIyX3EtDZuNFyeZhLiW0OmXP+aicdMOEACG2nsujtU/XQE+rdLn03rFojGH5MZUR7lLeC+Zs2OOgLzNXI9/ba+oAbjkE/ibX1r33KRL1faFTZZ1yeZHMZl9XFzOv8DsToQ2ohFSNEXOh8FrVUAWl6tK84rNFcfSeRCruHZXuDr5OEZB/yygyXS7UMn4b5wRK1YrCVgzs77SRtYqZ5QPejCfzc9d9AsZ6Men+crKkvNWKOGGZVdaNqyyLh6CD6eCjcd16x9tlXyTqEOwhqVDWaMgkUcnDJSX7YjuJH9ZSFBCttBJJFQMYQR3W6kRBmpCbnW792EUhLkrNwN8WeJ/bRdLR3IKoVCOe5feEzlnJwtWlxE/NjG74VNU6qZdpgXKxHpoBG7Es8grFtagsE/PGssHG6PMc6HCV5XFHsJeEHOMVuxM022zl+qBUVj7xgxqoj/For/tO0Fm2hYTEhbPGaq00LTQOsffBQKwtbb92j171jXHxw4jkhQZLtT7gQSWV+FcwZqYOb8vZra2LWyO+IQwQBK2KrZFHcpBbTxSYQt8nhu78SQ6OuRQL7eGaPyaUQHCHepBHafs6y0hqpalQxNFQnNk7bMonMXbideLktViE35ZsPo+UkDZIHIhuP2LqzAR5PXXMzKTg6sWQH1h6mAWrpWENB7SxoF0hRLgt7ohqSyNBriHSfiFAlbBNN54hQssw+H73jtEN+tzNyagNCTe6YbRTN/FqA23mhUwrqi/mlwZsAY1LN00yGgOUHERRn9KauWIhv0O6yhvWbuMO8OnlRdsLMZ7b2FE7g2Afvjwu8bAycMzs+m1vBJAHyhKPKyBXZCH0I/XDg+T4wXdeWVG+lsgo266qEklNczUB4pwCrax6BNV0HkU5tuBj7db5ijfrnMLIg594TEZUAtrfOGmPftf3J8GvWp7QSoaSapz/Ec1g0EcB4Ol2ZrjGV2wD93+8TD2e8jJvnKuojG/KP6rWdCn0Xy4sZKJk7tcs8u3u0Xa3+G/WudRMDZmn2TKQmKnJon1iTKdllOdTKVgxWf5E721ye40KN9A6AU+RyLV7t8oYFdv7WfasHrfTWB3bws4+5KeNosTNnJFt8/FL57KbtmMKjflsJgWr6IZLlRLRnW7fUYPZwi1oyyRHV38dNpKo6gcUcxgUKbOzLCdDwoFdDllcdZtKsstDQy6oFPecthV9Xu34xW/QGjTjZFHKhtzM2dEJaPxXke9bdifu2SnBfsunBEsnGvSCCqrM34Ji0OFER5c1tkUHtGwzg1PGrqpuessmxlR1LHWBZoVZsuOyGMTHfmDw3CZxttFwkm6g/4coUHaFI+JjvYFNnYK8I7BTXQZ5zPXirPZY3ZrLgvXNN8t8ztLGCKFYk1V6pRCd7Y8IvoFZtizoZToOnaqhbBngFphx2muHM7feCzK4EAnlAOJWM+zn2/fDzMHcMwFe3Iffchmqwn2pNCZRvlGGlv75TGtZJWsrp5XXpaE+MGUXrUL9BPwXQ/IYbfEVblt6bSzqSN06HSKlBcym6g5KHArp3Pi4vAkyaU0AV55PvketX2auZzfxTqZBDk2eTvkHLErPI8bQZ7+QIUX61s675Xfbv7pavMzkH+zShrqPIgqW5Lo1gT275X5eUn7rk5a0gPM0WJEesoIbsHD1srToKjWtfUa6Fj96buCuSFipwaQkAEnp7saHPZ0iOwnOWsqoZwEMvgcJtpS3mrq4/Egq73vPTYzZMWAuSZNyIOuJ9/iMWfedM/Cg2DCFtHbyWXfMqQG91yqweNMJqBuBXmRWmKPIDIg28iW1VYHgz14CHAqafbvuHDui0x+WjvRTgmRGJvlSef5RRtazDhUXfOUAsBFNLbDXtdfCGL6vIevZq1pGgFXmS01j8UO64aK0TQ3/PuxCEeLychhv1WC0hiP2q3ZbluCfjEo/qgIZWp5Vex5mSxBoQ1ChOjgw8G2pvkQY5JC4+ncfWCG5LJ1OnBVULJZ9txjT4Z+uBajUk8idG1cAKJCsbh8WPTdyy95L4eKuLQyseFgytiuf0+vUWnjk/TXD25UIWDoHjx4BGbCBKmu+wQCy9pbnfU/ug0JZtBcRwcKlqB4bIcJi1X4d9fnluCufOgbacmJ36HSk8ydkM8RR8Y2/W/ogUOB/J/xxafhpN5nu5PJIqVTJl11ZHPYyUYsrZGU+G1WBC6F4zJv7DxKqTzUfDOXUAoBnWSB3VRMbrHZenKGhtaoWn4h76cP5q+EZB+iiIp6hZOsg63j6YdxCF8ze0XYjyWzRKY9j0SSOZml+QEWCkO6XlWFnBBL4Wk9FLgxF6Ns6mGDa6LY/Og+OK9Lq2U0FM24AuXsWRwSNW+gDBHf7F0enCgH0N4p5X7BuSMAAACB4mjlw9y4JgAB3kGw0AEAkJISB7HEZ/sCAAAAAARZWg=='))
assert hashlib.sha256(_v32_source).hexdigest() == '4c61d512e78dbc87c8cb04a8e001a97a615e8156ff836b581d69344cb4735a9f'
_v32 = _types.ModuleType('v35_frozen_v32')
_v32.__dict__['previous'] = _v31
_v32_source = _v32_source.replace(b'import scripts.v31_notebook_core as previous', b'')
exec(compile(_v32_source, 'frozen-v32', 'exec'), _v32.__dict__)
del _v32_source
previous = _v32
del _v27, _v29, _v30, _v31, _v32


v31, v29, legacy = previous.previous, previous.v29, previous.legacy
EXPERIMENT = "v35_full_set_balanced_attention"
CONTROL_HASH = "71c863d9cb05f3efc63ac54dd7faa05efd2fc137a11da9adcd2064c82c88dfee"
MAX_HOURS = 10.75
CHECKPOINT_EPOCHS = (12, 24, 36, 48)
CONFIGS = (
    {"id": "long_geo", "kind": "attention", "geo": True, "width": 192, "seed": 20263501,
     "epochs": 48, "rare_branch": False, "negative_clip": .02, "negative_gamma": 4., "ranking_weight": 0.},
    {"id": "balanced_ecology", "kind": "balanced_attention", "geo": False, "width": 192, "seed": 20263502,
     "epochs": 48, "rare_branch": True, "negative_clip": .02, "negative_gamma": 2., "ranking_weight": .08},
    {"id": "balanced_geo", "kind": "balanced_attention", "geo": True, "width": 192, "seed": 20263503,
     "epochs": 48, "rare_branch": True, "negative_clip": .02, "negative_gamma": 4., "ranking_weight": .08})
ENSEMBLES = {"equal": (1., 1., 1.), "ecology_lean": (1., 2., 1.), "balanced_pair": (0., 1., 1.)}
POLICIES = tuple({"id": f"{name}_{mode}", "ensemble": name, "count_mode": mode}
                 for name in ENSEMBLES for mode in ("scale0.7", "scale1", "scale1.3", "fixed18", "fixed24"))
MIN_GAIN = .001
MIN_OUTSIDE_GAIN = .0005
MIN_CI = .00025


class BalancedAttention(v29.SensorAttention):
    """Pool each sensor separately so 64 image tokens cannot dominate 1 env token."""
    def __init__(self, env_dim, static_dim, species, width=192):
        super().__init__(env_dim, static_dim, species, width)
        self.fusion = nn.Sequential(nn.Linear(6*width, 2*width), nn.GELU(), nn.Dropout(.15),
                                    legacy.ResidualVectorBlock(2*width, .15))

    def forward(self, x, geo=True):
        image = self.image(x["sentinel"]).flatten(2).transpose(1, 2)
        land = self.land(x["landsat"].permute(0, 3, 1, 2).flatten(2))
        climate = self.climate(x["bioclim"].permute(0, 2, 1, 3).flatten(2))
        env, static = self.environment(x["environment"])[:, None], self.static(x["static_geo" if geo else "static_eco"])[:, None]
        sensors = [image, land, climate, env, static]
        if self.training:
            sensors = [s*torch.bernoulli(s.new_full((len(s), 1, 1), .9)) for s in sensors]
        tokens = torch.cat([self.cls.expand(len(image), -1, -1)]+sensors, 1)
        encoded = self.encoder(tokens+self.position[:, :tokens.shape[1]])
        pooled, offset = [encoded[:, 0]], 1
        for sensor in sensors:
            pooled.append(encoded[:, offset:offset+sensor.shape[1]].mean(1))
            offset += sensor.shape[1]
        feature = self.norm(self.fusion(torch.cat(pooled, 1)))
        return self.head(feature), self.richness(feature).squeeze(1)


def create_model(store, config, indices):
    if config["kind"] == "attention":
        model = v29.create_model(store, config, indices)
    else:
        legacy.set_seed(config["seed"])
        model = BalancedAttention(store.dims["environment"],
            store.static_train.shape[1] if config["geo"] else store.eco_train.shape[1], len(store.species_ids), config["width"])
        frequency = legacy._frequency(store.labels, indices)
        prior = np.clip((frequency+.5)/(len(indices)+1.), 1e-5, .95)
        with torch.no_grad():
            model.head.bias.copy_(torch.tensor(np.log(prior/(1-prior)), dtype=torch.float32))
            model.richness.bias.fill_(math.log1p(frequency.sum()/len(indices)))
    frequency = legacy._frequency(store.labels, indices)
    rare = np.flatnonzero((frequency >= 5) & (frequency/len(indices) <= .005))
    if config["rare_branch"] and len(rare):
        model.head = previous.RareResidualHead(model.head, rare)
    return model, rare


def ranking_loss(logits, target):
    """Each survey's positive mass sums to one; supports soft mixup targets.

    Auxiliary ranking surrogate, NOT an exact expected-F1 loss.
    """
    distribution = target.float()/target.sum(1, keepdim=True).clamp_min(1e-6)
    return -(distribution*F.log_softmax(logits.float(), dim=1)).sum(1).mean()


def checkpoint_score(probability, target, countries):
    rank, _ = legacy.top_rank(probability, min(30, probability.shape[1]))
    values = np.mean([legacy.f1_from_ranked(target, rank, np.full(len(rank), min(k, rank.shape[1])))
                      for k in (12, 18, 24, 30)], axis=0)
    frame = pd.DataFrame({"score": values, "country": np.asarray(countries).astype(str)})
    country = frame.groupby("country").score.agg(["mean", "size"])
    supported = country.loc[country["size"] >= 30, "mean"]
    outside = frame.loc[~frame.country.isin(v31.CORE_COUNTRIES), "score"]
    macro = float(supported.mean()) if len(supported) else float(values.mean())
    transfer = float(outside.mean()) if len(outside) else float(values.mean())
    return float(.5*values.mean()+.25*macro+.25*transfer)


def train_model(store, rows, indices, selection, stats, config, directory, guard, device, *, production_epochs=None):
    started = time.monotonic()
    phase = "production" if production_epochs is not None else "development"
    epochs = int(production_epochs or config["epochs"])
    if epochs > config["epochs"]:
        raise ValueError("Production schedule exceeds registered schedule")
    model, rare = create_model(store, config, indices)
    model = model.to(device); ema = copy.deepcopy(model).eval()
    for parameter in ema.parameters():
        parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=.02)
    # Keep the same learning-rate prefix when production stops at a selected epoch.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, config["epochs"], eta_min=1e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    rng, weights = np.random.default_rng(config["seed"]), v29.sample_weights(rows, indices)
    frequency = legacy._frequency(store.labels, indices)
    positive = torch.as_tensor(np.clip((20/np.maximum(frequency, 1))**.25, 1, 3), dtype=torch.float32, device=device)
    best, best_epoch, history = -np.inf, 0, []
    directory.mkdir(parents=True, exist_ok=True)
    checkpoint = directory/f"{phase}_{config['id']}.pt"
    batch = 48 if device.type == "cuda" else 16
    for epoch in range(1, epochs+1):
        start, total, seen = time.monotonic(), 0., 0
        uniform = rng.permutation(indices)[:len(indices)//2]
        order = rng.permutation(np.r_[uniform, rng.choice(indices, len(indices)-len(uniform), p=weights)])
        model.train()
        for begin in range(0, len(order), batch):
            guard.require(60*60, f"{phase} v35 training")
            take = order[begin:begin+batch]
            x = v29.batch_inputs(store, take, stats, device, augment=True, rng=rng)
            target = torch.as_tensor(np.asarray(store.labels[take], np.float32), device=device)
            if len(take) > 1 and rng.random() < .5:
                mix, permutation = float(rng.beta(.2, .2)), torch.randperm(len(take), device=device)
                x = {k: mix*v+(1-mix)*v[permutation] for k, v in x.items()}
                target = mix*target+(1-mix)*target[permutation]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits, richness = model(x, config["geo"])
                loss = 100*previous.asymmetric_loss(logits, target, positive, config["negative_clip"], config["negative_gamma"])
                loss += config["ranking_weight"]*ranking_loss(logits, target)
                loss += .04*F.smooth_l1_loss(richness.float(), torch.log1p(target.sum(1)))
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite v35 loss")
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 2.)
            scaler.step(optimizer); scaler.update()
            with torch.no_grad():
                for average, current in zip(ema.parameters(), model.parameters()):
                    average.lerp_(current, .02)
            total += float(loss.detach())*len(take); seen += len(take)
        scheduler.step()
        score = None
        if production_epochs is None and (epoch in CHECKPOINT_EPOCHS or epoch == epochs):
            p = v29.predict(ema, store, selection, stats, device, config, views=1, guard=guard)
            score = checkpoint_score(p, np.asarray(store.labels[selection]), rows.iloc[selection].country)
            if score > best:
                best, best_epoch = score, epoch
                torch.save(ema.state_dict(), checkpoint)
            del p
        record = {"epoch": epoch, "loss": total/max(seen, 1), "selection_score": score, "seconds": time.monotonic()-start,
                  "learning_rate_after_epoch": optimizer.param_groups[0]["lr"]}
        history.append(record); guard.stamp("v35_train", model=config["id"], phase=phase, **record)
    if production_epochs is not None:
        best_epoch = epochs
        torch.save(ema.state_dict(), checkpoint)
    if best_epoch < 1:
        raise RuntimeError("No selected checkpoint")
    ema.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    del model, optimizer
    previous.release(device)
    return ema, {"config": config, "training_surveys": len(indices), "best_epoch": best_epoch,
        "selection_score": float(best) if production_epochs is None else None, "history": history,
        "seconds": time.monotonic()-started, "parameters": sum(p.numel() for p in ema.parameters()),
        "rare_columns": len(rare), "all_species_outputs_trained": len(store.species_ids),
        "scheduler_total_epochs": config["epochs"], "checkpoint_sha256": legacy.sha256_file(checkpoint)}


def decode(probability, mode):
    maximum = min(40, probability.shape[1]); minimum = min(8, maximum)
    rank, value = legacy.top_rank(probability, maximum)
    if mode.startswith("fixed"):
        counts = np.full(len(probability), min(maximum, max(minimum, int(mode[5:]))))
    else:
        scale = float(mode[5:])
        k = np.arange(1, maximum+1)
        objective = 2*np.cumsum(value, 1)/(k[None]+probability.sum(1)[:, None]*scale+1e-8)
        counts = objective[:, minimum-1:].argmax(1)+minimum
    return [row[:count].astype(int).tolist() for row, count in zip(rank, counts)]


def combine(members, name):
    weights = np.asarray(ENSEMBLES[name], np.float32); weights /= weights.sum()
    result = np.zeros_like(members[0], dtype=np.float32)
    for weight, member in zip(weights, members):
        if weight:
            result += np.asarray(member, np.float32)*weight
    return result


def load_reference(payload):
    packed = base64.b64decode(payload, validate=True)
    if hashlib.sha256(packed).hexdigest() != REFERENCE_PAYLOAD_HASH:
        raise ValueError("Reference payload mismatch")
    raw = lzma.decompress(packed)
    if hashlib.sha256(raw).hexdigest() != REFERENCE_RAW_HASH:
        raise ValueError("Reference raw hash mismatch")
    return json.loads(raw)


def label_hash(labels):
    digest = hashlib.sha256()
    for start in range(0, len(labels), 1024):
        digest.update(np.asarray(labels[start:start+1024], np.uint8).tobytes())
    return digest.hexdigest()


def bind_reference(reference, rows, species, splits, manifests, labels):
    """Match ALL original training/evaluation IDs before trusting cached scores."""
    if len(splits) != 2 or len(manifests) != 2 or len(reference["folds"]) != 2:
        raise ValueError("Exactly two cached folds required")
    if hashlib.sha256(np.asarray(species, '<i8').tobytes()).hexdigest() != reference["species_sha256"]:
        raise ValueError("Reference species vocabulary mismatch")
    if label_hash(labels) != reference["labels_sha256"]:
        raise ValueError("Official PA labels changed; cached comparator is invalid")
    lookup = pd.Index(rows.surveyId)
    bound = []
    for f, (split, manifest, saved) in enumerate(zip(splits, manifests, reference["folds"])):
        if manifest["id_hashes"] != saved["id_hashes"]:
            raise ValueError("Cached baseline split IDs changed; do not compare incompatible training recipes")
        roles = {}
        for role in ("calibration", "assessment"):
            ids = np.asarray(saved[role]["ids"], np.int64)
            take = lookup.get_indexer(ids)
            score = np.asarray(saved[role]["f1"], np.float64)
            if ((take < 0).any() or len(np.unique(ids)) != len(ids) or len(score) != len(ids) or
                not np.isfinite(score).all() or np.any((score < 0) | (score > 1)) or
                not set(take).issubset(set(split[role]))):
                raise ValueError("Invalid cached comparator alignment")
            if role == "assessment" and set(take) != set(split[role]):
                raise ValueError("Incomplete cached assessment")
            if role == "calibration":
                prediction = saved[role]["predictions"]
                if not np.allclose(legacy.score_prediction_lists(np.asarray(labels[take]), prediction), score, atol=1e-10, rtol=0):
                    raise ValueError("Calibration labels/cache mismatch")
            roles[role] = {"indices": take, "reference_f1": score}
        bound.append(roles)
    if len(bound) != 2:
        raise ValueError("Exactly two cached folds required")
    return bound


def fit_fold(number, split, bound, rows, store, directory, guard, device):
    started = time.monotonic()
    stats = v29.fit_normalization(store, split["training"])
    data = {r: {**b, "members": []} for r, b in bound.items()}
    records, calibrations = [], []
    for source in CONFIGS:
        guard.require(3*3600, "start v35 development model")
        config = {**source, "seed": source["seed"]+number*1000}
        model, record = train_model(store, rows, split["training"], split["selection"], stats, config,
                                    directory/f"fold{number}", guard, device)
        p = v29.predict(model, store, split["selection"], stats, device, config, views=2, guard=guard)
        cal = v29.fit_platt(p, np.asarray(store.labels[split["selection"]]))
        calibrations.append(cal); records.append(record)
        for role, d in data.items():
            p = v29.predict(model, store, d["indices"], stats, device, config, views=2, guard=guard)
            d["members"].append(v29.calibrated(p, cal).astype(np.float16))
        del model, p
        previous.release(device)
    trials = {}
    d = data["calibration"]; target = np.asarray(store.labels[d["indices"]])
    for name in ENSEMBLES:
        probability = combine(d["members"], name)
        for policy in (p for p in POLICIES if p["ensemble"] == name):
            trials[policy["id"]] = legacy.score_prediction_lists(target, decode(probability, policy["count_mode"]))
        del probability
    return {"number": number, "split": split, "data": data, "records": records, "calibrations": calibrations,
        "trials": trials, "wall_seconds": time.monotonic()-started}


def select_policy(bundles, rows):
    take = np.concatenate([b["data"]["calibration"]["indices"] for b in bundles])
    reference = np.concatenate([b["data"]["calibration"]["reference_f1"] for b in bundles])
    trials = []
    for policy in POLICIES:
        scores = np.concatenate([b["trials"][policy["id"]] for b in bundles])
        gains = [float((b["trials"][policy["id"]]-b["data"]["calibration"]["reference_f1"]).mean()) for b in bundles]
        geo = v31.geographic_gain(scores-reference, rows.iloc[take].country, rows.iloc[take].surveyId)
        allowed = min(gains) > 0 and geo["gain"] >= MIN_GAIN and geo["outside_core_gain"] >= MIN_OUTSIDE_GAIN and geo["geographic_gain_positive"]
        trials.append({**policy, **geo, "fold_gains": gains, "allowed": bool(allowed), "sample_f1": float(scores.mean())})
    # If no policy passes, retain the best ATTEMPT for honest diagnostics, not a
    # hidden all-zero control report. It cannot authorize production/submission.
    eligible = [t for t in trials if t["allowed"]]
    chosen = max(eligible or trials, key=lambda t: (t["robust_gain"], t["gain"]))
    return next(dict(p) for p in POLICIES if p["id"] == chosen["id"]), trials, bool(eligible)


def regression_check(bundles, policy, rows, store):
    frames, summaries = [], []
    for b in bundles:
        d = b["data"]["assessment"]; take = d["indices"]
        probability = combine(d["members"], policy["ensemble"])
        prediction = decode(probability, policy["count_mode"])
        target = np.asarray(store.labels[take]); new = legacy.score_prediction_lists(target, prediction)
        old = d["reference_f1"]
        members = {c["id"]: float(legacy.score_prediction_lists(target, decode(p, policy["count_mode"])).mean())
                   for c, p in zip(CONFIGS, d["members"])}
        frames.append(pd.DataFrame({"surveyId": rows.iloc[take].surveyId.to_numpy(), "fold": b["number"],
            "country": rows.iloc[take].country.to_numpy(), "spatial_block": legacy.spatial_blocks(rows.iloc[take]),
            "cached_matched_v32_f1": old, "v35_f1": new, "delta_f1": new-old,
            "predicted_cardinality": list(map(len, prediction))}))
        summaries.append({"fold": b["number"], "gain": float((new-old).mean()),
            "members_not_used_for_policy_selection": members, "multilabel": v29.multilabel_summary(target, prediction),
            "species_groups": legacy.species_group_metrics(target, prediction, legacy._frequency(store.labels, b["split"]["training"]))})
        del probability
    frame = pd.concat(frames, ignore_index=True)
    if not frame.surveyId.is_unique:
        raise ValueError("Duplicate assessment surveys")
    return frame, {**v31.geographic_gain(frame.delta_f1, frame.country), "folds": summaries,
        "surveys": len(frame), "matched_v32_f1": float(frame.cached_matched_v32_f1.mean()), "v35_f1": float(frame.v35_f1.mean()),
        "bootstrap": legacy.paired_block_bootstrap(frame.delta_f1.to_numpy(), frame.spatial_block.to_numpy(), iterations=1000, seed=20263507),
        "fresh_assessment": False, "used_for_policy_selection": False,
        "by_country": legacy.summarize_by_group(frame, "country", ("cached_matched_v32_f1", "v35_f1", "delta_f1"))}


def production_epochs(bundles):
    return [int(np.median([b["records"][i]["best_epoch"] for b in bundles])) for i in range(len(CONFIGS))]


def production_estimate(bundles, full_rows, *, maximum=False):
    epochs = [c["epochs"] for c in CONFIGS] if maximum else production_epochs(bundles)
    return float(sum(max(np.median([h["seconds"] for h in b["records"][i]["history"]])/b["records"][i]["training_surveys"]
                    for b in bundles)*full_rows*epochs[i]*1.5 for i in range(len(CONFIGS)))+45*60)


def remaining_plan_estimate(bundles, next_training_rows, full_rows):
    first = bundles[0]
    return float(first["wall_seconds"]*max(1., next_training_rows/len(first["split"]["training"]))*1.25
                 +production_estimate(bundles, full_rows, maximum=True)+15*60)


def fit_production(bundles, policy, rows, test_rows, store, directory, guard, device):
    estimate = production_estimate(bundles, len(rows))
    guard.require(estimate, "whole v35 production admission")
    take = np.arange(len(rows)); stats = v29.fit_normalization(store, take)
    epochs, members, records = production_epochs(bundles), [], []
    for i, config in enumerate(CONFIGS):
        if ENSEMBLES[policy["ensemble"]][i] == 0:
            members.append(None); continue
        model, record = train_model(store, rows, take, np.array([], np.int64), stats, config, directory,
                                   guard, device, production_epochs=epochs[i])
        cal = {k: float(np.mean([b["calibrations"][i][k] for b in bundles])) for k in ("slope", "intercept")}
        p = v29.predict(model, store, np.arange(len(test_rows)), stats, device, config, test=True, views=2, guard=guard)
        members.append(v29.calibrated(p, cal).astype(np.float16))
        records.append({**record, "calibration": cal, "calibration_source": "selection-only fold coefficients; full-data transfer assumption"})
        del model, p; previous.release(device)
    # A zero-weight omitted member must not allocate an unnecessary N x S array.
    weights = np.asarray(ENSEMBLES[policy["ensemble"]], np.float32); weights /= weights.sum()
    probability = np.zeros((len(test_rows), len(store.species_ids)), np.float32)
    for p, weight in zip(members, weights):
        if weight:
            probability += np.asarray(p, np.float32)*weight
    prediction = decode(probability, policy["count_mode"])
    return prediction, records, {"training_rows": len(take), "all_PA_rows_used": True,
        "calibration_anchor_rows_removed": 0, "epochs_selected_before_assessment": epochs,
        "scheduler_prefix_preserved": True, "admission_estimate_seconds": estimate,
        "mean_cardinality": float(np.mean(list(map(len, prediction)))), "full_set_predictions": True}


def save_compact(path, bundles, rows, store, per_assessment=500):
    ids, folds, roles, countries, ranks, values, masses, reference, truths = [], [], [], [], [], [], [], [], []
    for b in bundles:
        for role in ("calibration", "assessment"):
            data = b["data"][role]; take = data["indices"]
            n = len(take) if role == "calibration" else min(per_assessment, len(take))
            pos = np.sort(np.random.default_rng(20263508+b["number"]).choice(len(take), n, replace=False,
                          p=v29.sample_weights(rows, take)))
            rs, vs, ms = [], [], []
            for member in data["members"]:
                p = np.asarray(member[pos], np.float32); rank, value = v31.compact_rank(p)
                rs.append(rank); vs.append(value); ms.append(p.sum(1))
            ranks.append(np.stack(rs, 1)); values.append(np.stack(vs, 1)); masses.append(np.stack(ms, 1))
            ids.append(rows.iloc[take[pos]].surveyId.to_numpy()); folds.append(np.full(n, b["number"], np.uint8))
            roles.append(np.full(n, role)); countries.append(rows.iloc[take[pos]].country.fillna("unknown").to_numpy(dtype="U64"))
            reference.append(data["reference_f1"][pos])
            truths.extend(np.flatnonzero(store.labels[j]).astype(np.uint16) for j in take[pos])
    np.savez_compressed(path, survey_id=np.concatenate(ids), fold=np.concatenate(folds), role=np.concatenate(roles),
        country=np.concatenate(countries), ranked_species_columns=np.concatenate(ranks), probability=np.concatenate(values),
        probability_mass=np.concatenate(masses), cached_matched_v32_f1=np.concatenate(reference),
        true_species_columns=np.concatenate(truths), true_offsets=np.r_[0, np.cumsum(list(map(len, truths)))].astype(np.uint32),
        species_ids=store.species_ids, expert_ids=np.array([c["id"] for c in CONFIGS]),
        evidence_scope=np.array("previously consumed calibration and assessment; individual model top128 plus FULL probability mass"))


def self_tests():
    logits = torch.zeros(2, 12, requires_grad=True)
    target = torch.zeros(2, 12); target[0, :2] = 1; target[1, :4] = .5
    loss = ranking_loss(logits, target); loss.backward()
    assert torch.isfinite(loss) and torch.isfinite(logits.grad).all()
    for p in POLICIES:
        output = decode(np.full((2, 50), .05, np.float32), p["count_mode"])
        assert all(8 <= len(x) <= 40 and len(x) == len(set(x)) for x in output)
    return {"passed": True, "tests": 4}


def publish(export, temporary, template, ids, prediction, species, gate):
    proof, name, differs = None, None, False
    if prediction is not None:
        path = temporary/"candidate.csv"
        proof = legacy.write_submission(path, template, ids, prediction, species)
        differs = proof["sha256"] != CONTROL_HASH
        if gate and differs:
            name = "GLC25_PA_submission_v35.csv"
            path.replace(export/name)
    decision = {"eligible_for_submission": name is not None, "prediction_file": name, "different_from_v32": differs,
        "message": "SUBMIT ONLY GLC25_PA_submission_v35.csv" if name else "NO NEW SUBMISSION; keep the scored v32"}
    if name is None:
        legacy.save_json(export/"NO_SUBMISSION.json", decision)
    return proof, decision


def run_v35(reference_b64):
    guard = legacy.RuntimeGuard(MAX_HOURS)
    working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
    temporary, export = working/"v35_runtime", working/"v35_export"
    legacy._clean_directory(temporary, working); legacy._clean_directory(export, working)
    temporary.mkdir(parents=True); export.mkdir(parents=True)
    store, bundles = None, []
    try:
        device = legacy.require_gpu()
        torch.set_num_threads(min(os.cpu_count() or 2, 6))
        tests, reference = self_tests(), load_reference(reference_b64)
        root = legacy.discover_data_root()
        if legacy.sha256_file(root/"GLC25_PA_metadata_train.csv") != reference["train_metadata_sha256"]:
            raise ValueError("Official training metadata changed; cached reference invalid")
        preflight = pd.read_csv(root/"GLC25_PA_metadata_train.csv", usecols=["surveyId", "lat", "lon", "country"])
        preflight = preflight.drop_duplicates("surveyId").sort_values("surveyId").reset_index(drop=True)
        _, manifests = v31.previous.make_splits(preflight)
        if len(manifests) != len(reference["folds"]) or any(m["id_hashes"] != s["id_hashes"] for m, s in zip(manifests, reference["folds"])):
            raise ValueError("Cached reference partitions do not match official metadata")
        guard.stamp("preflight", folds=manifests, cached_reference=True, fresh_assessment=False)
        original_writer = legacy._write_remote_arrays
        try:
            legacy._write_remote_arrays = v29.write_multiresolution
            features = legacy.prepare_feature_store(root, temporary/"features", guard, workers=6)
        finally:
            legacy._write_remote_arrays = original_writer
        store = legacy.FeatureStore(temporary/"features")
        store.high_train = np.load(store.cache/"train_sentinel64.npy", mmap_mode="r")
        store.high_test = np.load(store.cache/"test_sentinel64.npy", mmap_mode="r")
        rows, test_rows, pairs = legacy.load_rows_and_pairs(root, store.train_ids, store.test_ids)
        del pairs, preflight
        store.static_train, store.static_test = v29.candidate_static(rows), v29.candidate_static(test_rows)
        store.eco_train, store.eco_test = v29.candidate_static(rows, False), v29.candidate_static(test_rows, False)
        splits, manifests = v31.previous.make_splits(rows)
        bound = bind_reference(reference, rows, store.species_ids, splits, manifests, store.labels)
        template = pd.read_csv(root/"GLC25_SAMPLE_SUBMISSION.csv")
        for i, split in enumerate(splits):
            if bundles:
                estimate = remaining_plan_estimate(bundles, len(split["training"]), len(rows))
                guard.stamp("remaining_plan_admission", estimate_seconds=estimate)
                guard.require(estimate, "remaining development and longest production plan")
            bundles.append(fit_fold(i, split, bound[i], rows, store, temporary, guard, device))
        policy, trials, calibration_pass = select_policy(bundles, rows)
        epochs = production_epochs(bundles)
        legacy.save_json(temporary/"frozen_policy.json", {"policy": policy, "trials": trials, "production_epochs": epochs})
        guard.stamp("policy_frozen", policy=policy, calibration_pass=calibration_pass, production_epochs=epochs)
        frame, regression = regression_check(bundles, policy, rows, store)
        gates = {"calibration_pass": calibration_pass,
            "positive_each_fold": all(f["gain"] > 0 for f in regression["folds"]),
            "minimum_practical_gain": regression["gain"] >= MIN_GAIN,
            "spatial_ci_margin": regression["bootstrap"]["ci95"][0] >= MIN_CI,
            "geographic_transfer_positive": regression["geographic_gain_positive"],
            "minimum_outside_core_gain": regression["outside_core_gain"] >= MIN_OUTSIDE_GAIN,
            "geographic_buffers": min(m["minimum_distance_km"] for m in manifests) >= 20,
            "cached_reference_ids_and_labels_verified": True}
        prediction, fitted, diagnostics = None, [], {"skipped": "development gates failed; actual candidate diagnostics retained"}
        if all(gates.values()):
            prediction, fitted, diagnostics = fit_production(bundles, policy, rows, test_rows, store, temporary, guard, device)
        gates["within_budget"] = guard.elapsed_hours() < MAX_HOURS
        proof, decision = publish(export, temporary, template, store.test_ids, prediction, store.species_ids, all(gates.values()))
        frame.to_csv(export/"regression_per_survey_v35.csv", index=False)
        save_compact(export/"predictions_top128_v35.npz", bundles, rows, store)
        report = {"experiment": EXPERIMENT, "status": "complete", **decision, "runtime_hours": guard.elapsed_hours(),
            "self_tests": tests, "policy": policy, "calibration_trials": trials, "gates": gates,
            "regression": regression, "submission_validation": proof, "official_submission_made": False,
            "training": {"development": [{"fold": b["number"], "members": b["records"], "calibrations": b["calibrations"],
                "wall_seconds": b["wall_seconds"]} for b in bundles], "production": fitted},
            "production_diagnostics": diagnostics, "production_epochs_frozen_before_assessment": epochs,
            "cached_reference": {"provenance": reference["provenance"], "payload_hash": REFERENCE_PAYLOAD_HASH,
                "reference_model_refits_this_run": 0, "calibration_rows_per_fold": [len(b["calibration"]["indices"]) for b in bound]},
            "hardware": {"gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None, "torch": torch.__version__},
            "train_country_counts": rows.country.value_counts().to_dict(), "test_country_counts_unlabeled": test_rows.country.value_counts().to_dict(),
            "validation_status": "all 88,987 PA surveys consumed before; repeated spatial development, no independent holdout",
            "limitations": ["No guarantee of hidden-test improvement, winning, or full 12-hour completion.",
                "Full-set replacement may regress despite passing these repeated development gates.",
                "Calibration uses the 3000 archived sampled fold views, not the full historical calibration pool.",
                "The cached comparator is the unchanged v32 RECIPE refit inside the v34 run, not its test score.",
                "New pooling, auxiliary ranking, training duration, and decoding change together; component causality is not established.",
                "Countries with no PA labels, including much of Ukraine and the UK, cannot be independently validated here.",
                "Production uses every PA row; calibration transfers from smaller folds and can shift.",
                "The ratio-of-expectations count decoder and ranking loss are surrogates, not exact expected-F1 optimization."]}
        legacy.save_json(export/"v35_report.json", report)
        manifest = {"experiment": EXPERIMENT, "source_sha256": V35_SOURCE_HASH, "embedded_sources_sha256": FROZEN_SOURCE_HASHES,
            "splits": manifests, "features": features, "configs": CONFIGS, "checkpoint_epochs": CHECKPOINT_EPOCHS,
            "ensembles": ENSEMBLES, "policies": POLICIES, "reference_sha256": REFERENCE_PAYLOAD_HASH,
            "control_submission_sha256": CONTROL_HASH, "runtime_cap_hours": MAX_HOURS,
            "gate_margins": {"gain": MIN_GAIN, "outside_core": MIN_OUTSIDE_GAIN, "ci_lower": MIN_CI},
            "no_external_data_or_weights": True, "fresh_assessment": False,
            "outputs": {p.name: legacy.sha256_file(p) for p in sorted(export.iterdir())}}
        legacy.save_json(export/"v35_manifest.json", manifest)
        size = sum(p.stat().st_size for p in export.iterdir())
        if len(list(export.iterdir())) != 5 or size > 16_000_000:
            raise ValueError("Compact five-file/16MB export contract exceeded")
        guard.require(0, "final export")
        return {"status": "complete", **decision, "runtime_hours": guard.elapsed_hours(), "output_bytes": size,
            "export_directory": str(export), "regression_gain": regression["gain"], "fresh_assessment": False, "policy": policy}
    except Exception as error:
        ready = export/"GLC25_PA_submission_v35.csv"
        if ready.exists():
            ready.replace(export/"failed_predictions_DO_NOT_SUBMIT.txt")
        report_path = export/"v35_report.json"
        if report_path.exists():
            failed = json.loads(report_path.read_text(encoding="utf-8"))
            failed.update(status="failed", eligible_for_submission=False, prediction_file=None, message="DO NOT SUBMIT; export failed")
            legacy.save_json(report_path, failed)
        manifest_path = export/"v35_manifest.json"
        if manifest_path.exists():
            manifest_path.replace(export/"failed_manifest_not_a_valid_export.json")
        legacy.save_json(export/"failure_report.json", {"experiment": EXPERIMENT, "status": "failed", "error": str(error),
            "traceback": traceback.format_exc(), "runtime_hours": guard.elapsed_hours(), "official_submission_made": False,
            "eligible_for_submission": False, "prediction_file": None,
            "completed_folds": [{"fold": b["number"], "records": b["records"], "wall_seconds": b["wall_seconds"]} for b in bundles]})
        raise
    finally:
        del store
        previous.release(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        legacy._clean_directory(temporary, working)


In [ ]:
V35_SOURCE_HASH = 'afcd2f16466bcbd4ce17b9f93729e2be99754144b97ade7af68e990fedf0635d'
FROZEN_SOURCE_HASHES = {'v27': '69a2d3952fa6324d9a05f4edb6fa7662388a198df89c407f7f034b94099f1dc0', 'v29': 'bdb162dbc822622d34e9cdc0c4f9e91ee89fd7e11c33ad3113b4b6adccb5f3e1', 'v30': 'faf8f4d9b106097a5de7f11a156f4ea99cee809815970f2d802ae7f72634bc03', 'v31': '35b414a5a804ab2eae663ae619d564c5014094d081e01531cec6f781bf934fdb', 'v32': '4c61d512e78dbc87c8cb04a8e001a97a615e8156ff836b581d69344cb4735a9f'}
REFERENCE_PAYLOAD_HASH = 'dab2a85d76dac12df9581e43370cab811d36541f8f19658360c58b370148f885'
REFERENCE_RAW_HASH = '1d0753d6068d9bece7ce897f1ea8bf8becd80fd3cb16896cc8f779408f665e10'
REFERENCE_B64 = '...'
print({'source_sha256':V35_SOURCE_HASH,'cached_reference':'unchanged v32 recipe from v34 run','fresh_holdout_remaining':0})


In [ ]:
V35_RESULT = run_v35(REFERENCE_B64)
print(json.dumps(V35_RESULT,indent=2))
print(V35_RESULT['message'])
V35_RESULT
